In [0]:
# ============================================================
# SILVER LAYER - IMPORT REQUIRED LIBRARIES
# ============================================================

# PySpark functions for data cleaning and transformation
from pyspark.sql import functions as F

# Window functions are used for deduplication
from pyspark.sql.window import Window

In [0]:
# ============================================================
# BRONZE STORAGE PATHS
# ============================================================

# ADLS Gen2 Bronze path containing MSSQL tables
bronze_mssql_path = (
    "abfss://bronze@stshowroomanalytics01.dfs.core.windows.net/mssql/"
)

# ADLS Gen2 Bronze path containing PostgreSQL tables
bronze_postgresql_path = (
    "abfss://bronze@stshowroomanalytics01.dfs.core.windows.net/postgresql/"
)

# Display the paths so we can verify the configuration
print("MSSQL Bronze Path:", bronze_mssql_path)
print("PostgreSQL Bronze Path:", bronze_postgresql_path)

MSSQL Bronze Path: abfss://bronze@stshowroomanalytics01.dfs.core.windows.net/mssql/
PostgreSQL Bronze Path: abfss://bronze@stshowroomanalytics01.dfs.core.windows.net/postgresql/


In [0]:
# ============================================================
# VERIFY BRONZE STORAGE
# ============================================================

# List MSSQL folders available in Bronze
print("MSSQL Bronze folders:")
display(dbutils.fs.ls(bronze_mssql_path))

# List PostgreSQL folders available in Bronze
print("PostgreSQL Bronze folders:")
display(dbutils.fs.ls(bronze_postgresql_path))

MSSQL Bronze folders:


path,name,size,modificationTime
abfss://bronze@stshowroomanalytics01.dfs.core.windows.net/mssql/campaign/,campaign/,0,1786906955000
abfss://bronze@stshowroomanalytics01.dfs.core.windows.net/mssql/campaign_expense/,campaign_expense/,0,1786906966000
abfss://bronze@stshowroomanalytics01.dfs.core.windows.net/mssql/customer/,customer/,0,1786906966000
abfss://bronze@stshowroomanalytics01.dfs.core.windows.net/mssql/customer_followup/,customer_followup/,0,1786906955000
abfss://bronze@stshowroomanalytics01.dfs.core.windows.net/mssql/expenses/,expenses/,0,1786906962000
abfss://bronze@stshowroomanalytics01.dfs.core.windows.net/mssql/inventory/,inventory/,0,1786906958000
abfss://bronze@stshowroomanalytics01.dfs.core.windows.net/mssql/marketing_lead/,marketing_lead/,0,1786906961000
abfss://bronze@stshowroomanalytics01.dfs.core.windows.net/mssql/sales/,sales/,0,1786906963000
abfss://bronze@stshowroomanalytics01.dfs.core.windows.net/mssql/salesperson/,salesperson/,0,1786906970000
abfss://bronze@stshowroomanalytics01.dfs.core.windows.net/mssql/showroom/,showroom/,0,1786906955000


PostgreSQL Bronze folders:


path,name,size,modificationTime
abfss://bronze@stshowroomanalytics01.dfs.core.windows.net/postgresql/vehicle/,vehicle/,0,1786906989000
abfss://bronze@stshowroomanalytics01.dfs.core.windows.net/postgresql/vehicle_model/,vehicle_model/,0,1786906961000


In [0]:
# ============================================================
# SHOWROOM - READ BRONZE DATA
# ============================================================

# Path to the showroom Bronze data
showroom_path = (
    bronze_mssql_path + "showroom/"
)

# Read the Parquet files into a Spark DataFrame
df_showroom = spark.read.parquet(showroom_path)

# Display the Bronze data
display(df_showroom)

# Display schema
df_showroom.printSchema()

# Display record count
print("Total showroom records:", df_showroom.count())

showroom_id,showroom_name,brand,city,state,manager_name,opening_date,created_date,updated_date
1,Pune Central,Tata,Pune,Maharashtra,Rahul Patil,2020-01-15,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z
2,Mumbai Andheri,Hyundai,Mumbai,Maharashtra,Amit Sharma,2019-06-10,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z
3,Nashik Road,Maruti,Nashik,Maharashtra,Priya Deshmukh,2021-03-20,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z
4,Nagpur Central,Mahindra,Nagpur,Maharashtra,Vikas Jadhav,2020-08-12,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z
5,Thane West,Kia,Thane,Maharashtra,Sneha Kulkarni,2022-01-05,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z
6,Navi Mumbai,Toyota,Navi Mumbai,Maharashtra,Rohit Joshi,2021-09-18,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z
7,Aurangabad City,Honda,Aurangabad,Maharashtra,Sagar More,2020-11-25,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z
8,Kolhapur Central,Skoda,Kolhapur,Maharashtra,Neha Patil,2022-04-15,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z
9,Ahmednagar Road,Volkswagen,Ahmednagar,Maharashtra,Kunal Shinde,2021-07-10,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z
10,Solapur Central,Tata,Solapur,Maharashtra,Pooja Pawar,2022-06-20,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z


root
 |-- showroom_id: integer (nullable = true)
 |-- showroom_name: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- manager_name: string (nullable = true)
 |-- opening_date: date (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- updated_date: timestamp (nullable = true)

Total showroom records: 15


In [0]:
# ============================================================
# SHOWROOM - DATA QUALITY CHECK
# ============================================================

# Count NULL values in each column
df_showroom.select([
    F.sum(F.col(c).isNull().cast("int")).alias(f"{c}_nulls")
    for c in df_showroom.columns
]).show()

# Check whether showroom_id contains duplicates
print("Duplicate showroom IDs:")

df_showroom.groupBy("showroom_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

+-----------------+-------------------+-----------+----------+-----------+------------------+------------------+------------------+------------------+
|showroom_id_nulls|showroom_name_nulls|brand_nulls|city_nulls|state_nulls|manager_name_nulls|opening_date_nulls|created_date_nulls|updated_date_nulls|
+-----------------+-------------------+-----------+----------+-----------+------------------+------------------+------------------+------------------+
|                0|                  0|          0|         0|          0|                 0|                 0|                 0|                 0|
+-----------------+-------------------+-----------+----------+-----------+------------------+------------------+------------------+------------------+

Duplicate showroom IDs:
+-----------+-----+
|showroom_id|count|
+-----------+-----+
+-----------+-----+



In [0]:
# ============================================================
# SHOWROOM - CLEAN STRING COLUMNS
# ============================================================

# Remove unnecessary leading/trailing spaces
df_showroom_clean = (
    df_showroom
    .withColumn("showroom_name", F.trim(F.col("showroom_name")))
    .withColumn("brand", F.trim(F.col("brand")))
    .withColumn("city", F.trim(F.col("city")))
    .withColumn("state", F.trim(F.col("state")))
    .withColumn("manager_name", F.trim(F.col("manager_name")))
)

# Standardize text casing
df_showroom_clean = (
    df_showroom_clean
    .withColumn("showroom_name", F.initcap(F.col("showroom_name")))
    .withColumn("brand", F.initcap(F.col("brand")))
    .withColumn("city", F.initcap(F.col("city")))
    .withColumn("state", F.upper(F.col("state")))
    .withColumn("manager_name", F.initcap(F.col("manager_name")))
)

display(df_showroom_clean)

showroom_id,showroom_name,brand,city,state,manager_name,opening_date,created_date,updated_date
1,Pune Central,Tata,Pune,MAHARASHTRA,Rahul Patil,2020-01-15,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z
2,Mumbai Andheri,Hyundai,Mumbai,MAHARASHTRA,Amit Sharma,2019-06-10,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z
3,Nashik Road,Maruti,Nashik,MAHARASHTRA,Priya Deshmukh,2021-03-20,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z
4,Nagpur Central,Mahindra,Nagpur,MAHARASHTRA,Vikas Jadhav,2020-08-12,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z
5,Thane West,Kia,Thane,MAHARASHTRA,Sneha Kulkarni,2022-01-05,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z
6,Navi Mumbai,Toyota,Navi Mumbai,MAHARASHTRA,Rohit Joshi,2021-09-18,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z
7,Aurangabad City,Honda,Aurangabad,MAHARASHTRA,Sagar More,2020-11-25,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z
8,Kolhapur Central,Skoda,Kolhapur,MAHARASHTRA,Neha Patil,2022-04-15,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z
9,Ahmednagar Road,Volkswagen,Ahmednagar,MAHARASHTRA,Kunal Shinde,2021-07-10,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z
10,Solapur Central,Tata,Solapur,MAHARASHTRA,Pooja Pawar,2022-06-20,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z


In [0]:
# ============================================================
# SHOWROOM - REMOVE INVALID RECORDS
# ============================================================

# showroom_id is the business key.
# A record without showroom_id cannot be reliably identified.
df_showroom_clean = (
    df_showroom_clean
    .filter(F.col("showroom_id").isNotNull())
)

print(
    "Records after invalid-record filtering:",
    df_showroom_clean.count()
)

Records after invalid-record filtering: 15


In [0]:
# ============================================================
# SHOWROOM - DEDUPLICATION
# ============================================================

# Keep the latest record for each showroom_id.
# updated_date determines which record is the newest.
showroom_window = (
    Window
    .partitionBy("showroom_id")
    .orderBy(F.col("updated_date").desc_nulls_last())
)

df_showroom_clean = (
    df_showroom_clean
    .withColumn(
        "_row_number",
        F.row_number().over(showroom_window)
    )
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

print(
    "Records after deduplication:",
    df_showroom_clean.count()
)

Records after deduplication: 15


In [0]:
# ============================================================
# SHOWROOM - SILVER PROCESSING METADATA
# ============================================================

# Add the timestamp when the record was processed by Silver.
df_showroom_clean = df_showroom_clean.withColumn(
    "_silver_processed_timestamp",
    F.current_timestamp()
)

display(df_showroom_clean)

showroom_id,showroom_name,brand,city,state,manager_name,opening_date,created_date,updated_date,_silver_processed_timestamp
1,Pune Central,Tata,Pune,MAHARASHTRA,Rahul Patil,2020-01-15,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z,2026-08-17T07:54:30.371Z
2,Mumbai Andheri,Hyundai,Mumbai,MAHARASHTRA,Amit Sharma,2019-06-10,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z,2026-08-17T07:54:30.371Z
3,Nashik Road,Maruti,Nashik,MAHARASHTRA,Priya Deshmukh,2021-03-20,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z,2026-08-17T07:54:30.371Z
4,Nagpur Central,Mahindra,Nagpur,MAHARASHTRA,Vikas Jadhav,2020-08-12,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z,2026-08-17T07:54:30.371Z
5,Thane West,Kia,Thane,MAHARASHTRA,Sneha Kulkarni,2022-01-05,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z,2026-08-17T07:54:30.371Z
6,Navi Mumbai,Toyota,Navi Mumbai,MAHARASHTRA,Rohit Joshi,2021-09-18,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z,2026-08-17T07:54:30.371Z
7,Aurangabad City,Honda,Aurangabad,MAHARASHTRA,Sagar More,2020-11-25,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z,2026-08-17T07:54:30.371Z
8,Kolhapur Central,Skoda,Kolhapur,MAHARASHTRA,Neha Patil,2022-04-15,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z,2026-08-17T07:54:30.371Z
9,Ahmednagar Road,Volkswagen,Ahmednagar,MAHARASHTRA,Kunal Shinde,2021-07-10,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z,2026-08-17T07:54:30.371Z
10,Solapur Central,Tata,Solapur,MAHARASHTRA,Pooja Pawar,2022-06-20,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z,2026-08-17T07:54:30.371Z


In [0]:
# ============================================================
# SHOWROOM - WRITE TO SILVER DELTA
# ============================================================

# Write the cleaned showroom data as a Delta table.
# The table belongs to:
# Catalog = showroom_analytics
# Schema  = silver
# Table   = showroom

df_showroom_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "showroom_analytics.silver.showroom"
    )

In [0]:
# ============================================================
# SHOWROOM - VERIFY SILVER TABLE
# ============================================================

# Read the newly created Silver Delta table
df_silver_showroom = spark.table(
    "showroom_analytics.silver.showroom"
)

# Display Silver records
display(df_silver_showroom)

# Display Silver schema
df_silver_showroom.printSchema()

# Display final record count
print(
    "Silver showroom records:",
    df_silver_showroom.count()
)

showroom_id,showroom_name,brand,city,state,manager_name,opening_date,created_date,updated_date,_silver_processed_timestamp
1,Pune Central,Tata,Pune,MAHARASHTRA,Rahul Patil,2020-01-15,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z,2026-08-17T07:55:34.132Z
2,Mumbai Andheri,Hyundai,Mumbai,MAHARASHTRA,Amit Sharma,2019-06-10,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z,2026-08-17T07:55:34.132Z
3,Nashik Road,Maruti,Nashik,MAHARASHTRA,Priya Deshmukh,2021-03-20,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z,2026-08-17T07:55:34.132Z
4,Nagpur Central,Mahindra,Nagpur,MAHARASHTRA,Vikas Jadhav,2020-08-12,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z,2026-08-17T07:55:34.132Z
5,Thane West,Kia,Thane,MAHARASHTRA,Sneha Kulkarni,2022-01-05,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z,2026-08-17T07:55:34.132Z
6,Navi Mumbai,Toyota,Navi Mumbai,MAHARASHTRA,Rohit Joshi,2021-09-18,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z,2026-08-17T07:55:34.132Z
7,Aurangabad City,Honda,Aurangabad,MAHARASHTRA,Sagar More,2020-11-25,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z,2026-08-17T07:55:34.132Z
8,Kolhapur Central,Skoda,Kolhapur,MAHARASHTRA,Neha Patil,2022-04-15,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z,2026-08-17T07:55:34.132Z
9,Ahmednagar Road,Volkswagen,Ahmednagar,MAHARASHTRA,Kunal Shinde,2021-07-10,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z,2026-08-17T07:55:34.132Z
10,Solapur Central,Tata,Solapur,MAHARASHTRA,Pooja Pawar,2022-06-20,2026-08-15T16:06:49.114Z,2026-08-15T16:06:49.114Z,2026-08-17T07:55:34.132Z


root
 |-- showroom_id: integer (nullable = true)
 |-- showroom_name: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- manager_name: string (nullable = true)
 |-- opening_date: date (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- updated_date: timestamp (nullable = true)
 |-- _silver_processed_timestamp: timestamp (nullable = true)

Silver showroom records: 15


In [0]:
# ============================================================
# SALESPERSON - READ BRONZE DATA
# ============================================================

# Build the path to the salesperson Bronze folder
salesperson_path = (
    bronze_mssql_path + "salesperson/"
)

# Read Parquet files from Bronze
df_salesperson = spark.read.parquet(salesperson_path)

# Display the source data
display(df_salesperson)

# Display schema
df_salesperson.printSchema()

# Display record count
print(
    "Total salesperson records:",
    df_salesperson.count()
)

salesperson_id,salesperson_name,showroom_id,joining_date,created_date,updated_date
1,Rahul Patil,1,2022-04-10,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z
2,Amit Sharma,1,2023-01-15,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z
3,Priya Deshmukh,2,2021-07-20,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z
4,Vikas Jadhav,2,2023-03-12,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z
5,Sneha Kulkarni,3,2022-09-05,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z
6,ROHIT JOSHI,3,2024-01-18,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z
7,Sagar More,4,2021-11-25,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z
8,Neha Patil,4,2023-06-10,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z
9,Kunal Shinde,5,2022-02-14,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z
10,Pooja Pawar,5,2024-04-01,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z


root
 |-- salesperson_id: integer (nullable = true)
 |-- salesperson_name: string (nullable = true)
 |-- showroom_id: integer (nullable = true)
 |-- joining_date: date (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- updated_date: timestamp (nullable = true)

Total salesperson records: 15


In [0]:
# ============================================================
# SALESPERSON - DATA QUALITY CHECK
# ============================================================

# Count NULL values for every column
df_salesperson.select([
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(f"{c}_nulls")
    for c in df_salesperson.columns
]).show()

# Check duplicate salesperson IDs
print("Duplicate salesperson IDs:")

df_salesperson.groupBy("salesperson_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

+--------------------+----------------------+-----------------+------------------+------------------+------------------+
|salesperson_id_nulls|salesperson_name_nulls|showroom_id_nulls|joining_date_nulls|created_date_nulls|updated_date_nulls|
+--------------------+----------------------+-----------------+------------------+------------------+------------------+
|                   0|                     0|                0|                 0|                 0|                 0|
+--------------------+----------------------+-----------------+------------------+------------------+------------------+

Duplicate salesperson IDs:
+--------------+-----+
|salesperson_id|count|
+--------------+-----+
+--------------+-----+



In [0]:
# ============================================================
# SALESPERSON - CLEAN STRING DATA
# ============================================================

# Remove unnecessary spaces and standardize the salesperson name
df_salesperson_clean = (
    df_salesperson
    .withColumn(
        "salesperson_name",
        F.initcap(
            F.trim(
                F.col("salesperson_name")
            )
        )
    )
)

display(df_salesperson_clean)

salesperson_id,salesperson_name,showroom_id,joining_date,created_date,updated_date
1,Rahul Patil,1,2022-04-10,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z
2,Amit Sharma,1,2023-01-15,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z
3,Priya Deshmukh,2,2021-07-20,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z
4,Vikas Jadhav,2,2023-03-12,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z
5,Sneha Kulkarni,3,2022-09-05,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z
6,Rohit Joshi,3,2024-01-18,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z
7,Sagar More,4,2021-11-25,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z
8,Neha Patil,4,2023-06-10,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z
9,Kunal Shinde,5,2022-02-14,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z
10,Pooja Pawar,5,2024-04-01,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z


In [0]:
# ============================================================
# SALESPERSON - REMOVE INVALID RECORDS
# ============================================================

# salesperson_id is the business key.
# Records without an ID cannot be uniquely identified.
df_salesperson_clean = (
    df_salesperson_clean
    .filter(
        F.col("salesperson_id").isNotNull()
    )
)

print(
    "Records after invalid-record filtering:",
    df_salesperson_clean.count()
)

Records after invalid-record filtering: 15


In [0]:
# ============================================================
# SALESPERSON - DEDUPLICATION
# ============================================================

# Keep the latest version of each salesperson record.
# updated_date determines the latest version.
salesperson_window = (
    Window
    .partitionBy("salesperson_id")
    .orderBy(
        F.col("updated_date").desc_nulls_last()
    )
)

df_salesperson_clean = (
    df_salesperson_clean
    .withColumn(
        "_row_number",
        F.row_number().over(salesperson_window)
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)

print(
    "Records after deduplication:",
    df_salesperson_clean.count()
)

Records after deduplication: 15


In [0]:
# ============================================================
# SALESPERSON - SILVER PROCESSING METADATA
# ============================================================

# Record when this data was processed by the Silver layer.
df_salesperson_clean = df_salesperson_clean.withColumn(
    "_silver_processed_timestamp",
    F.current_timestamp()
)

display(df_salesperson_clean)

salesperson_id,salesperson_name,showroom_id,joining_date,created_date,updated_date,_silver_processed_timestamp
1,Rahul Patil,1,2022-04-10,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z,2026-08-17T07:57:37.564Z
2,Amit Sharma,1,2023-01-15,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z,2026-08-17T07:57:37.564Z
3,Priya Deshmukh,2,2021-07-20,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z,2026-08-17T07:57:37.564Z
4,Vikas Jadhav,2,2023-03-12,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z,2026-08-17T07:57:37.564Z
5,Sneha Kulkarni,3,2022-09-05,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z,2026-08-17T07:57:37.564Z
6,Rohit Joshi,3,2024-01-18,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z,2026-08-17T07:57:37.564Z
7,Sagar More,4,2021-11-25,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z,2026-08-17T07:57:37.564Z
8,Neha Patil,4,2023-06-10,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z,2026-08-17T07:57:37.564Z
9,Kunal Shinde,5,2022-02-14,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z,2026-08-17T07:57:37.564Z
10,Pooja Pawar,5,2024-04-01,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z,2026-08-17T07:57:37.564Z


In [0]:
# ============================================================
# SALESPERSON - WRITE TO SILVER DELTA
# ============================================================

# Save the cleaned salesperson data as a Delta table.
#
# Catalog = showroom_analytics
# Schema  = silver
# Table   = salesperson

df_salesperson_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "showroom_analytics.silver.salesperson"
    )

In [0]:
# ============================================================
# SALESPERSON - VERIFY SILVER TABLE
# ============================================================

# Read the newly created Silver Delta table
df_silver_salesperson = spark.table(
    "showroom_analytics.silver.salesperson"
)

# Display the Silver data
display(df_silver_salesperson)

# Display schema
df_silver_salesperson.printSchema()

# Display final record count
print(
    "Silver salesperson records:",
    df_silver_salesperson.count()
)

salesperson_id,salesperson_name,showroom_id,joining_date,created_date,updated_date,_silver_processed_timestamp
1,Rahul Patil,1,2022-04-10,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z,2026-08-17T07:57:45.794Z
2,Amit Sharma,1,2023-01-15,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z,2026-08-17T07:57:45.794Z
3,Priya Deshmukh,2,2021-07-20,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z,2026-08-17T07:57:45.794Z
4,Vikas Jadhav,2,2023-03-12,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z,2026-08-17T07:57:45.794Z
5,Sneha Kulkarni,3,2022-09-05,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z,2026-08-17T07:57:45.794Z
6,Rohit Joshi,3,2024-01-18,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z,2026-08-17T07:57:45.794Z
7,Sagar More,4,2021-11-25,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z,2026-08-17T07:57:45.794Z
8,Neha Patil,4,2023-06-10,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z,2026-08-17T07:57:45.794Z
9,Kunal Shinde,5,2022-02-14,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z,2026-08-17T07:57:45.794Z
10,Pooja Pawar,5,2024-04-01,2026-08-15T16:15:42.521Z,2026-08-15T16:15:42.521Z,2026-08-17T07:57:45.794Z


root
 |-- salesperson_id: integer (nullable = true)
 |-- salesperson_name: string (nullable = true)
 |-- showroom_id: integer (nullable = true)
 |-- joining_date: date (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- updated_date: timestamp (nullable = true)
 |-- _silver_processed_timestamp: timestamp (nullable = true)

Silver salesperson records: 15


In [0]:
# ============================================================
# CUSTOMER - READ BRONZE DATA
# ============================================================

# Build the path to the customer Bronze folder
customer_path = (
    bronze_mssql_path + "customer/"
)

# Read customer Parquet files from Bronze
df_customer = spark.read.parquet(customer_path)

# Display the source data
display(df_customer)

# Display schema
df_customer.printSchema()

# Display record count
print(
    "Total customer records:",
    df_customer.count()
)

customer_id,customer_name,phone,email,city,age,gender,occupation,created_date,updated_date
1,Sachin Kaware,9876543210,sachin.kaware@gmail.com,Pune,27,Male,Engineer,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
2,Amit Sharma,9876543211,amit.sharma@gmail.com,Mumbai,32,Male,Business,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
3,Priya Deshmukh,9876543212,priya.d@gmail.com,Nashik,29,Female,Teacher,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
4,Rahul Patil,9876543213,rahul.patil@gmail.com,Pune,35,Male,Business,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
5,Sneha Kulkarni,9876543214,sneha.k@gmail.com,Thane,28,Female,Engineer,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
6,ROHIT JOSHI,9876543215,rohit.joshi@gmail.com,Mumbai,41,Male,Doctor,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
7,Neha Patil,9876543216,neha.patil@gmail.com,Nagpur,31,Female,Government,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
8,Kunal Shinde,9876543217,kunal.shinde@gmail.com,Kolhapur,26,Male,Engineer,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
9,Pooja Pawar,9876543218,pooja.pawar@gmail.com,Aurangabad,34,Female,Teacher,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
10,Sagar More,9876543219,sagar.more@gmail.com,Nashik,39,Male,Business,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z


root
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- updated_date: timestamp (nullable = true)

Total customer records: 20


In [0]:
# ============================================================
# CUSTOMER - DATA QUALITY CHECK
# ============================================================

# Count NULL values for every customer column
df_customer.select([
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(f"{c}_nulls")
    for c in df_customer.columns
]).show()

# Check exact duplicate rows
total_customer_records = df_customer.count()

distinct_customer_records = (
    df_customer.dropDuplicates().count()
)

print(
    "Exact duplicate records:",
    total_customer_records - distinct_customer_records
)

+-----------------+-------------------+-----------+-----------+----------+---------+------------+----------------+------------------+------------------+
|customer_id_nulls|customer_name_nulls|phone_nulls|email_nulls|city_nulls|age_nulls|gender_nulls|occupation_nulls|created_date_nulls|updated_date_nulls|
+-----------------+-------------------+-----------+-----------+----------+---------+------------+----------------+------------------+------------------+
|                0|                  0|          1|          2|         0|        0|           0|               1|                 0|                 0|
+-----------------+-------------------+-----------+-----------+----------+---------+------------+----------------+------------------+------------------+

Exact duplicate records: 0


In [0]:
# ============================================================
# CUSTOMER - CHECK POTENTIAL BUSINESS KEY
# ============================================================

# Display the first few records so we can understand
# the customer identifier and important attributes.
display(
    df_customer.limit(20)
)

customer_id,customer_name,phone,email,city,age,gender,occupation,created_date,updated_date
1,Sachin Kaware,9876543210,sachin.kaware@gmail.com,Pune,27,Male,Engineer,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
2,Amit Sharma,9876543211,amit.sharma@gmail.com,Mumbai,32,Male,Business,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
3,Priya Deshmukh,9876543212,priya.d@gmail.com,Nashik,29,Female,Teacher,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
4,Rahul Patil,9876543213,rahul.patil@gmail.com,Pune,35,Male,Business,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
5,Sneha Kulkarni,9876543214,sneha.k@gmail.com,Thane,28,Female,Engineer,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
6,ROHIT JOSHI,9876543215,rohit.joshi@gmail.com,Mumbai,41,Male,Doctor,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
7,Neha Patil,9876543216,neha.patil@gmail.com,Nagpur,31,Female,Government,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
8,Kunal Shinde,9876543217,kunal.shinde@gmail.com,Kolhapur,26,Male,Engineer,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
9,Pooja Pawar,9876543218,pooja.pawar@gmail.com,Aurangabad,34,Female,Teacher,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
10,Sagar More,9876543219,sagar.more@gmail.com,Nashik,39,Male,Business,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z


In [0]:
# ============================================================
# CUSTOMER - DATA QUALITY CHECK
# ============================================================

# Count NULL values in every column
df_customer.select([
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(f"{c}_nulls")
    for c in df_customer.columns
]).show()

# Check duplicate customer IDs
print("Duplicate customer IDs:")

df_customer.groupBy("customer_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

+-----------------+-------------------+-----------+-----------+----------+---------+------------+----------------+------------------+------------------+
|customer_id_nulls|customer_name_nulls|phone_nulls|email_nulls|city_nulls|age_nulls|gender_nulls|occupation_nulls|created_date_nulls|updated_date_nulls|
+-----------------+-------------------+-----------+-----------+----------+---------+------------+----------------+------------------+------------------+
|                0|                  0|          1|          2|         0|        0|           0|               1|                 0|                 0|
+-----------------+-------------------+-----------+-----------+----------+---------+------------+----------------+------------------+------------------+

Duplicate customer IDs:
+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [0]:
# ============================================================
# CUSTOMER - CLEAN AND STANDARDIZE DATA
# ============================================================

df_customer_clean = (
    df_customer

    # Remove unnecessary spaces from customer name
    .withColumn(
        "customer_name",
        F.initcap(F.trim(F.col("customer_name")))
    )

    # Remove spaces around phone numbers
    .withColumn(
        "phone",
        F.trim(F.col("phone"))
    )

    # Convert email addresses to lowercase
    .withColumn(
        "email",
        F.lower(F.trim(F.col("email")))
    )

    # Standardize city
    .withColumn(
        "city",
        F.initcap(F.trim(F.col("city")))
    )

    # Standardize gender
    .withColumn(
        "gender",
        F.initcap(F.trim(F.col("gender")))
    )

    # Standardize occupation
    .withColumn(
        "occupation",
        F.initcap(F.trim(F.col("occupation")))
    )
)

display(df_customer_clean)

customer_id,customer_name,phone,email,city,age,gender,occupation,created_date,updated_date
1,Sachin Kaware,9876543210,sachin.kaware@gmail.com,Pune,27,Male,Engineer,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
2,Amit Sharma,9876543211,amit.sharma@gmail.com,Mumbai,32,Male,Business,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
3,Priya Deshmukh,9876543212,priya.d@gmail.com,Nashik,29,Female,Teacher,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
4,Rahul Patil,9876543213,rahul.patil@gmail.com,Pune,35,Male,Business,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
5,Sneha Kulkarni,9876543214,sneha.k@gmail.com,Thane,28,Female,Engineer,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
6,Rohit Joshi,9876543215,rohit.joshi@gmail.com,Mumbai,41,Male,Doctor,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
7,Neha Patil,9876543216,neha.patil@gmail.com,Nagpur,31,Female,Government,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
8,Kunal Shinde,9876543217,kunal.shinde@gmail.com,Kolhapur,26,Male,Engineer,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
9,Pooja Pawar,9876543218,pooja.pawar@gmail.com,Aurangabad,34,Female,Teacher,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z
10,Sagar More,9876543219,sagar.more@gmail.com,Nashik,39,Male,Business,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z


In [0]:
# ============================================================
# CUSTOMER - VALIDATE RECORDS
# ============================================================

# Remove records without a customer ID
df_customer_clean = (
    df_customer_clean
    .filter(F.col("customer_id").isNotNull())
)

# Keep age values that are either NULL or within a reasonable
# human age range. Invalid ages are converted to NULL rather
# than deleting the complete customer record.
df_customer_clean = (
    df_customer_clean
    .withColumn(
        "age",
        F.when(
            (F.col("age").isNull()) |
            ((F.col("age") >= 18) & (F.col("age") <= 100)),
            F.col("age")
        ).otherwise(F.lit(None).cast("int"))
    )
)

print(
    "Records after validation:",
    df_customer_clean.count()
)

Records after validation: 20


In [0]:
# ============================================================
# CUSTOMER - DEDUPLICATION
# ============================================================

# Create a window for each customer.
# The latest updated_date receives row number 1.
customer_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        F.col("updated_date").desc_nulls_last()
    )
)

# Keep only the latest customer record
df_customer_clean = (
    df_customer_clean
    .withColumn(
        "_row_number",
        F.row_number().over(customer_window)
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)

print(
    "Records after deduplication:",
    df_customer_clean.count()
)

Records after deduplication: 20


In [0]:
# ============================================================
# CUSTOMER - SILVER PROCESSING METADATA
# ============================================================

# Record the time when the Silver transformation processed
# the customer record.
df_customer_clean = df_customer_clean.withColumn(
    "_silver_processed_timestamp",
    F.current_timestamp()
)

display(df_customer_clean)

customer_id,customer_name,phone,email,city,age,gender,occupation,created_date,updated_date,_silver_processed_timestamp
1,Sachin Kaware,9876543210,sachin.kaware@gmail.com,Pune,27,Male,Engineer,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z,2026-08-17T08:00:12.917Z
2,Amit Sharma,9876543211,amit.sharma@gmail.com,Mumbai,32,Male,Business,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z,2026-08-17T08:00:12.917Z
3,Priya Deshmukh,9876543212,priya.d@gmail.com,Nashik,29,Female,Teacher,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z,2026-08-17T08:00:12.917Z
4,Rahul Patil,9876543213,rahul.patil@gmail.com,Pune,35,Male,Business,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z,2026-08-17T08:00:12.917Z
5,Sneha Kulkarni,9876543214,sneha.k@gmail.com,Thane,28,Female,Engineer,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z,2026-08-17T08:00:12.917Z
6,Rohit Joshi,9876543215,rohit.joshi@gmail.com,Mumbai,41,Male,Doctor,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z,2026-08-17T08:00:12.917Z
7,Neha Patil,9876543216,neha.patil@gmail.com,Nagpur,31,Female,Government,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z,2026-08-17T08:00:12.917Z
8,Kunal Shinde,9876543217,kunal.shinde@gmail.com,Kolhapur,26,Male,Engineer,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z,2026-08-17T08:00:12.917Z
9,Pooja Pawar,9876543218,pooja.pawar@gmail.com,Aurangabad,34,Female,Teacher,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z,2026-08-17T08:00:12.917Z
10,Sagar More,9876543219,sagar.more@gmail.com,Nashik,39,Male,Business,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z,2026-08-17T08:00:12.917Z


In [0]:
# ============================================================
# CUSTOMER - WRITE TO SILVER DELTA
# ============================================================

# Save the cleaned customer data as a Delta table.
#
# Catalog = showroom_analytics
# Schema  = silver
# Table   = customer

df_customer_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "showroom_analytics.silver.customer"
    )

In [0]:
# ============================================================
# CUSTOMER - VERIFY SILVER TABLE
# ============================================================

# Read the newly created Silver Delta table
df_silver_customer = spark.table(
    "showroom_analytics.silver.customer"
)

# Display Silver data
display(df_silver_customer)

# Display schema
df_silver_customer.printSchema()

# Display final record count
print(
    "Silver customer records:",
    df_silver_customer.count()
)

customer_id,customer_name,phone,email,city,age,gender,occupation,created_date,updated_date,_silver_processed_timestamp
1,Sachin Kaware,9876543210,sachin.kaware@gmail.com,Pune,27,Male,Engineer,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z,2026-08-17T08:00:23.577Z
2,Amit Sharma,9876543211,amit.sharma@gmail.com,Mumbai,32,Male,Business,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z,2026-08-17T08:00:23.577Z
3,Priya Deshmukh,9876543212,priya.d@gmail.com,Nashik,29,Female,Teacher,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z,2026-08-17T08:00:23.577Z
4,Rahul Patil,9876543213,rahul.patil@gmail.com,Pune,35,Male,Business,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z,2026-08-17T08:00:23.577Z
5,Sneha Kulkarni,9876543214,sneha.k@gmail.com,Thane,28,Female,Engineer,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z,2026-08-17T08:00:23.577Z
6,Rohit Joshi,9876543215,rohit.joshi@gmail.com,Mumbai,41,Male,Doctor,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z,2026-08-17T08:00:23.577Z
7,Neha Patil,9876543216,neha.patil@gmail.com,Nagpur,31,Female,Government,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z,2026-08-17T08:00:23.577Z
8,Kunal Shinde,9876543217,kunal.shinde@gmail.com,Kolhapur,26,Male,Engineer,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z,2026-08-17T08:00:23.577Z
9,Pooja Pawar,9876543218,pooja.pawar@gmail.com,Aurangabad,34,Female,Teacher,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z,2026-08-17T08:00:23.577Z
10,Sagar More,9876543219,sagar.more@gmail.com,Nashik,39,Male,Business,2026-08-15T16:15:42.550Z,2026-08-15T16:15:42.550Z,2026-08-17T08:00:23.577Z


root
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- updated_date: timestamp (nullable = true)
 |-- _silver_processed_timestamp: timestamp (nullable = true)

Silver customer records: 20


In [0]:
# ============================================================
# CAMPAIGN - READ BRONZE DATA
# ============================================================

# Path to the campaign Bronze folder
campaign_path = (
    bronze_mssql_path + "campaign/"
)

# Read campaign Parquet files from Bronze
df_campaign = spark.read.parquet(campaign_path)

# Display source data
display(df_campaign)

# Display schema
df_campaign.printSchema()

# Display record count
print(
    "Total campaign records:",
    df_campaign.count()
)

campaign_id,campaign_name,campaign_type,start_date,end_date,budget,created_date,updated_date
1,Diwali Mega Sale,Festival,2025-10-01,2025-11-15,1500000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
2,Google Search Campaign,Digital,2025-11-01,2025-12-31,900000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
3,Instagram SUV Campaign,Social Media,2025-12-01,2026-01-31,700000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
4,New Year Offer,Festival,2026-01-01,2026-01-31,1200000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
5,Republic Day Offer,Festival,2026-01-15,2026-01-31,800000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
6,Summer SUV Sale,Seasonal,2026-03-01,2026-04-30,1100000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
7,YouTube Campaign,Digital,2026-03-15,2026-05-15,650000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
8,Exchange Bonus,Promotion,2026-04-01,2026-05-31,950000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
9,Monsoon Offer,Seasonal,2026-06-01,2026-07-31,850000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
10,Independence Sale,Festival,2026-08-01,2026-08-20,1000000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z


root
 |-- campaign_id: integer (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- campaign_type: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- budget: decimal(18,2) (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- updated_date: timestamp (nullable = true)

Total campaign records: 10


In [0]:
# ============================================================
# CAMPAIGN - DATA QUALITY CHECK
# ============================================================

# Count NULL values in every column
df_campaign.select([
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(f"{c}_nulls")
    for c in df_campaign.columns
]).show()

# Check exact duplicate rows
total_campaign_records = df_campaign.count()

distinct_campaign_records = (
    df_campaign.dropDuplicates().count()
)

print(
    "Exact duplicate records:",
    total_campaign_records - distinct_campaign_records
)

+-----------------+-------------------+-------------------+----------------+--------------+------------+------------------+------------------+
|campaign_id_nulls|campaign_name_nulls|campaign_type_nulls|start_date_nulls|end_date_nulls|budget_nulls|created_date_nulls|updated_date_nulls|
+-----------------+-------------------+-------------------+----------------+--------------+------------+------------------+------------------+
|                0|                  0|                  0|               0|             0|           0|                 0|                 0|
+-----------------+-------------------+-------------------+----------------+--------------+------------+------------------+------------------+

Exact duplicate records: 0


In [0]:
# ============================================================
# CAMPAIGN - INSPECT SOURCE DATA
# ============================================================

# Display a sample of campaign records
display(
    df_campaign.limit(20)
)

campaign_id,campaign_name,campaign_type,start_date,end_date,budget,created_date,updated_date
1,Diwali Mega Sale,Festival,2025-10-01,2025-11-15,1500000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
2,Google Search Campaign,Digital,2025-11-01,2025-12-31,900000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
3,Instagram SUV Campaign,Social Media,2025-12-01,2026-01-31,700000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
4,New Year Offer,Festival,2026-01-01,2026-01-31,1200000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
5,Republic Day Offer,Festival,2026-01-15,2026-01-31,800000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
6,Summer SUV Sale,Seasonal,2026-03-01,2026-04-30,1100000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
7,YouTube Campaign,Digital,2026-03-15,2026-05-15,650000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
8,Exchange Bonus,Promotion,2026-04-01,2026-05-31,950000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
9,Monsoon Offer,Seasonal,2026-06-01,2026-07-31,850000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
10,Independence Sale,Festival,2026-08-01,2026-08-20,1000000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z


In [0]:
# ============================================================
# CAMPAIGN - DATA QUALITY CHECK
# ============================================================

# Count NULL values in every column
df_campaign.select([
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(f"{c}_nulls")
    for c in df_campaign.columns
]).show()

# Check duplicate campaign IDs
print("Duplicate campaign IDs:")

df_campaign.groupBy("campaign_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

+-----------------+-------------------+-------------------+----------------+--------------+------------+------------------+------------------+
|campaign_id_nulls|campaign_name_nulls|campaign_type_nulls|start_date_nulls|end_date_nulls|budget_nulls|created_date_nulls|updated_date_nulls|
+-----------------+-------------------+-------------------+----------------+--------------+------------+------------------+------------------+
|                0|                  0|                  0|               0|             0|           0|                 0|                 0|
+-----------------+-------------------+-------------------+----------------+--------------+------------+------------------+------------------+

Duplicate campaign IDs:
+-----------+-----+
|campaign_id|count|
+-----------+-----+
+-----------+-----+



In [0]:
# ============================================================
# CAMPAIGN - CLEAN STRING COLUMNS
# ============================================================

# Remove unnecessary spaces and standardize text fields
df_campaign_clean = (
    df_campaign
    .withColumn(
        "campaign_name",
        F.initcap(F.trim(F.col("campaign_name")))
    )
    .withColumn(
        "campaign_type",
        F.initcap(F.trim(F.col("campaign_type")))
    )
)

display(df_campaign_clean)

campaign_id,campaign_name,campaign_type,start_date,end_date,budget,created_date,updated_date
1,Diwali Mega Sale,Festival,2025-10-01,2025-11-15,1500000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
2,Google Search Campaign,Digital,2025-11-01,2025-12-31,900000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
3,Instagram Suv Campaign,Social Media,2025-12-01,2026-01-31,700000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
4,New Year Offer,Festival,2026-01-01,2026-01-31,1200000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
5,Republic Day Offer,Festival,2026-01-15,2026-01-31,800000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
6,Summer Suv Sale,Seasonal,2026-03-01,2026-04-30,1100000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
7,Youtube Campaign,Digital,2026-03-15,2026-05-15,650000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
8,Exchange Bonus,Promotion,2026-04-01,2026-05-31,950000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
9,Monsoon Offer,Seasonal,2026-06-01,2026-07-31,850000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z
10,Independence Sale,Festival,2026-08-01,2026-08-20,1000000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z


In [0]:
# ============================================================
# CAMPAIGN - VALIDATE RECORDS
# ============================================================

# campaign_id is the business key and therefore mandatory
df_campaign_clean = (
    df_campaign_clean
    .filter(F.col("campaign_id").isNotNull())
)

# A campaign should have a valid date range.
# We don't delete the record; instead, identify invalid ranges.
invalid_campaign_dates = (
    df_campaign_clean
    .filter(
        F.col("start_date").isNotNull() &
        F.col("end_date").isNotNull() &
        (F.col("end_date") < F.col("start_date"))
    )
)

print(
    "Campaigns with invalid date ranges:",
    invalid_campaign_dates.count()
)

display(invalid_campaign_dates)

Campaigns with invalid date ranges: 0


campaign_id,campaign_name,campaign_type,start_date,end_date,budget,created_date,updated_date


In [0]:
# ============================================================
# CAMPAIGN - VALIDATE BUDGET
# ============================================================

# Identify campaigns with negative budgets.
# Negative budget values are considered invalid.
invalid_campaign_budget = (
    df_campaign_clean
    .filter(
        F.col("budget").isNotNull() &
        (F.col("budget") < 0)
    )
)

print(
    "Campaigns with negative budget:",
    invalid_campaign_budget.count()
)

display(invalid_campaign_budget)

Campaigns with negative budget: 0


campaign_id,campaign_name,campaign_type,start_date,end_date,budget,created_date,updated_date


In [0]:
# ============================================================
# CAMPAIGN - DEDUPLICATION
# ============================================================

# Keep the latest version of each campaign.
# updated_date determines the latest record.
campaign_window = (
    Window
    .partitionBy("campaign_id")
    .orderBy(
        F.col("updated_date").desc_nulls_last()
    )
)

df_campaign_clean = (
    df_campaign_clean
    .withColumn(
        "_row_number",
        F.row_number().over(campaign_window)
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)

print(
    "Records after deduplication:",
    df_campaign_clean.count()
)

Records after deduplication: 10


In [0]:
# ============================================================
# CAMPAIGN - SILVER PROCESSING METADATA
# ============================================================

# Record when the Silver layer processed the campaign record.
df_campaign_clean = df_campaign_clean.withColumn(
    "_silver_processed_timestamp",
    F.current_timestamp()
)

display(df_campaign_clean)

campaign_id,campaign_name,campaign_type,start_date,end_date,budget,created_date,updated_date,_silver_processed_timestamp
1,Diwali Mega Sale,Festival,2025-10-01,2025-11-15,1500000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z,2026-08-17T08:03:16.811Z
2,Google Search Campaign,Digital,2025-11-01,2025-12-31,900000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z,2026-08-17T08:03:16.811Z
3,Instagram Suv Campaign,Social Media,2025-12-01,2026-01-31,700000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z,2026-08-17T08:03:16.811Z
4,New Year Offer,Festival,2026-01-01,2026-01-31,1200000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z,2026-08-17T08:03:16.811Z
5,Republic Day Offer,Festival,2026-01-15,2026-01-31,800000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z,2026-08-17T08:03:16.811Z
6,Summer Suv Sale,Seasonal,2026-03-01,2026-04-30,1100000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z,2026-08-17T08:03:16.811Z
7,Youtube Campaign,Digital,2026-03-15,2026-05-15,650000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z,2026-08-17T08:03:16.811Z
8,Exchange Bonus,Promotion,2026-04-01,2026-05-31,950000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z,2026-08-17T08:03:16.811Z
9,Monsoon Offer,Seasonal,2026-06-01,2026-07-31,850000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z,2026-08-17T08:03:16.811Z
10,Independence Sale,Festival,2026-08-01,2026-08-20,1000000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z,2026-08-17T08:03:16.811Z


In [0]:
# ============================================================
# CAMPAIGN - WRITE TO SILVER DELTA
# ============================================================

# Save the cleaned campaign data as a Delta table.
#
# Catalog = showroom_analytics
# Schema  = silver
# Table   = campaign

df_campaign_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "showroom_analytics.silver.campaign"
    )

In [0]:
# ============================================================
# CAMPAIGN - VERIFY SILVER TABLE
# ============================================================

# Read the newly created Silver Delta table
df_silver_campaign = spark.table(
    "showroom_analytics.silver.campaign"
)

# Display Silver data
display(df_silver_campaign)

# Display schema
df_silver_campaign.printSchema()

# Display final record count
print(
    "Silver campaign records:",
    df_silver_campaign.count()
)

campaign_id,campaign_name,campaign_type,start_date,end_date,budget,created_date,updated_date,_silver_processed_timestamp
1,Diwali Mega Sale,Festival,2025-10-01,2025-11-15,1500000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z,2026-08-17T08:04:45.479Z
2,Google Search Campaign,Digital,2025-11-01,2025-12-31,900000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z,2026-08-17T08:04:45.479Z
3,Instagram Suv Campaign,Social Media,2025-12-01,2026-01-31,700000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z,2026-08-17T08:04:45.479Z
4,New Year Offer,Festival,2026-01-01,2026-01-31,1200000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z,2026-08-17T08:04:45.479Z
5,Republic Day Offer,Festival,2026-01-15,2026-01-31,800000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z,2026-08-17T08:04:45.479Z
6,Summer Suv Sale,Seasonal,2026-03-01,2026-04-30,1100000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z,2026-08-17T08:04:45.479Z
7,Youtube Campaign,Digital,2026-03-15,2026-05-15,650000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z,2026-08-17T08:04:45.479Z
8,Exchange Bonus,Promotion,2026-04-01,2026-05-31,950000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z,2026-08-17T08:04:45.479Z
9,Monsoon Offer,Seasonal,2026-06-01,2026-07-31,850000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z,2026-08-17T08:04:45.479Z
10,Independence Sale,Festival,2026-08-01,2026-08-20,1000000.00,2026-08-15T16:15:42.569Z,2026-08-15T16:15:42.569Z,2026-08-17T08:04:45.479Z


root
 |-- campaign_id: integer (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- campaign_type: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- budget: decimal(18,2) (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- updated_date: timestamp (nullable = true)
 |-- _silver_processed_timestamp: timestamp (nullable = true)

Silver campaign records: 10


In [0]:
# ============================================================
# MARKETING LEAD - READ BRONZE DATA
# ============================================================

# Path to the marketing_lead Bronze folder
marketing_lead_path = (
    bronze_mssql_path + "marketing_lead/"
)

# Read Parquet files from Bronze
df_marketing_lead = spark.read.parquet(
    marketing_lead_path
)

# Display the source data
display(df_marketing_lead)

# Display schema
df_marketing_lead.printSchema()

# Display source record count
print(
    "Total marketing lead records:",
    df_marketing_lead.count()
)

lead_id,customer_id,showroom_id,campaign_id,salesperson_id,lead_date,lead_source,lead_status,created_date,updated_date
1,1,1,2,1,2026-01-05,Google Ads,Converted,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
2,2,2,3,3,2026-01-12,Instagram,Qualified,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
3,3,3,1,5,2026-01-15,Walk-in,New,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
4,4,1,4,2,2026-01-18,google ads,Contacted,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
5,5,5,3,9,2026-01-22,instagram,Qualified,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
6,6,2,5,4,2026-01-25,Website,Converted,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
7,7,4,6,7,2026-02-01,Facebook,Lost,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
8,8,5,7,10,2026-02-05,Referral,Contacted,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
9,9,7,8,13,2026-02-10,Google Ads,Qualified,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
10,10,3,9,6,2026-02-15,Website,Converted,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z


root
 |-- lead_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- showroom_id: integer (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- salesperson_id: integer (nullable = true)
 |-- lead_date: date (nullable = true)
 |-- lead_source: string (nullable = true)
 |-- lead_status: string (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- updated_date: timestamp (nullable = true)

Total marketing lead records: 20


In [0]:
# ============================================================
# MARKETING LEAD - DATA QUALITY CHECK
# ============================================================

# Count NULL values in every column
df_marketing_lead.select([
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(f"{c}_nulls")
    for c in df_marketing_lead.columns
]).show()

# Check exact duplicate rows
total_records = df_marketing_lead.count()

distinct_records = (
    df_marketing_lead.dropDuplicates().count()
)

print(
    "Exact duplicate records:",
    total_records - distinct_records
)

+-------------+-----------------+-----------------+-----------------+--------------------+---------------+-----------------+-----------------+------------------+------------------+
|lead_id_nulls|customer_id_nulls|showroom_id_nulls|campaign_id_nulls|salesperson_id_nulls|lead_date_nulls|lead_source_nulls|lead_status_nulls|created_date_nulls|updated_date_nulls|
+-------------+-----------------+-----------------+-----------------+--------------------+---------------+-----------------+-----------------+------------------+------------------+
|            0|                0|                0|                0|                   0|              0|                0|                0|                 0|                 0|
+-------------+-----------------+-----------------+-----------------+--------------------+---------------+-----------------+-----------------+------------------+------------------+

Exact duplicate records: 0


In [0]:
# ============================================================
# MARKETING LEAD - INSPECT SOURCE DATA
# ============================================================

# Display a sample of marketing lead records
display(
    df_marketing_lead.limit(20)
)

lead_id,customer_id,showroom_id,campaign_id,salesperson_id,lead_date,lead_source,lead_status,created_date,updated_date
1,1,1,2,1,2026-01-05,Google Ads,Converted,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
2,2,2,3,3,2026-01-12,Instagram,Qualified,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
3,3,3,1,5,2026-01-15,Walk-in,New,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
4,4,1,4,2,2026-01-18,google ads,Contacted,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
5,5,5,3,9,2026-01-22,instagram,Qualified,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
6,6,2,5,4,2026-01-25,Website,Converted,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
7,7,4,6,7,2026-02-01,Facebook,Lost,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
8,8,5,7,10,2026-02-05,Referral,Contacted,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
9,9,7,8,13,2026-02-10,Google Ads,Qualified,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
10,10,3,9,6,2026-02-15,Website,Converted,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z


In [0]:
# ============================================================
# MARKETING LEAD - DATA QUALITY CHECK
# ============================================================

# Count NULL values in every column
df_marketing_lead.select([
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(f"{c}_nulls")
    for c in df_marketing_lead.columns
]).show()

# Check duplicate lead IDs
print("Duplicate lead IDs:")

df_marketing_lead.groupBy("lead_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

+-------------+-----------------+-----------------+-----------------+--------------------+---------------+-----------------+-----------------+------------------+------------------+
|lead_id_nulls|customer_id_nulls|showroom_id_nulls|campaign_id_nulls|salesperson_id_nulls|lead_date_nulls|lead_source_nulls|lead_status_nulls|created_date_nulls|updated_date_nulls|
+-------------+-----------------+-----------------+-----------------+--------------------+---------------+-----------------+-----------------+------------------+------------------+
|            0|                0|                0|                0|                   0|              0|                0|                0|                 0|                 0|
+-------------+-----------------+-----------------+-----------------+--------------------+---------------+-----------------+-----------------+------------------+------------------+

Duplicate lead IDs:
+-------+-----+
|lead_id|count|
+-------+-----+
+-------+-----+



In [0]:
# ============================================================
# MARKETING LEAD - CLEAN STRING COLUMNS
# ============================================================

df_marketing_lead_clean = (
    df_marketing_lead

    # Standardize lead source
    .withColumn(
        "lead_source",
        F.initcap(
            F.trim(
                F.col("lead_source")
            )
        )
    )

    # Standardize lead status
    .withColumn(
        "lead_status",
        F.initcap(
            F.trim(
                F.col("lead_status")
            )
        )
    )
)

display(df_marketing_lead_clean)

lead_id,customer_id,showroom_id,campaign_id,salesperson_id,lead_date,lead_source,lead_status,created_date,updated_date
1,1,1,2,1,2026-01-05,Google Ads,Converted,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
2,2,2,3,3,2026-01-12,Instagram,Qualified,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
3,3,3,1,5,2026-01-15,Walk-in,New,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
4,4,1,4,2,2026-01-18,Google Ads,Contacted,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
5,5,5,3,9,2026-01-22,Instagram,Qualified,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
6,6,2,5,4,2026-01-25,Website,Converted,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
7,7,4,6,7,2026-02-01,Facebook,Lost,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
8,8,5,7,10,2026-02-05,Referral,Contacted,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
9,9,7,8,13,2026-02-10,Google Ads,Qualified,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z
10,10,3,9,6,2026-02-15,Website,Converted,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z


In [0]:
# ============================================================
# MARKETING LEAD - VALIDATE RECORDS
# ============================================================

# lead_id is the business key and must be present.
df_marketing_lead_clean = (
    df_marketing_lead_clean
    .filter(
        F.col("lead_id").isNotNull()
    )
)

print(
    "Records after lead ID validation:",
    df_marketing_lead_clean.count()
)

Records after lead ID validation: 20


In [0]:
# ============================================================
# MARKETING LEAD - REFERENTIAL INTEGRITY CHECK
# ============================================================

# Load the existing Silver dimension/reference tables
df_customer_silver = spark.table(
    "showroom_analytics.silver.customer"
)

df_showroom_silver = spark.table(
    "showroom_analytics.silver.showroom"
)

df_campaign_silver = spark.table(
    "showroom_analytics.silver.campaign"
)

df_salesperson_silver = spark.table(
    "showroom_analytics.silver.salesperson"
)

# Check customer IDs that do not exist in Silver customer
invalid_customers = (
    df_marketing_lead_clean
    .join(
        df_customer_silver.select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
)

print(
    "Leads with invalid customer_id:",
    invalid_customers.count()
)

# Check showroom IDs that do not exist
invalid_showrooms = (
    df_marketing_lead_clean
    .join(
        df_showroom_silver.select("showroom_id"),
        on="showroom_id",
        how="left_anti"
    )
)

print(
    "Leads with invalid showroom_id:",
    invalid_showrooms.count()
)

# Check campaign IDs that do not exist
invalid_campaigns = (
    df_marketing_lead_clean
    .join(
        df_campaign_silver.select("campaign_id"),
        on="campaign_id",
        how="left_anti"
    )
)

print(
    "Leads with invalid campaign_id:",
    invalid_campaigns.count()
)

# Check salesperson IDs that do not exist
invalid_salespersons = (
    df_marketing_lead_clean
    .join(
        df_salesperson_silver.select("salesperson_id"),
        on="salesperson_id",
        how="left_anti"
    )
)

print(
    "Leads with invalid salesperson_id:",
    invalid_salespersons.count()
)

Leads with invalid customer_id: 0
Leads with invalid showroom_id: 0
Leads with invalid campaign_id: 0
Leads with invalid salesperson_id: 0


In [0]:
# ============================================================
# MARKETING LEAD - DEDUPLICATION
# ============================================================

# Keep the latest version of each lead.
lead_window = (
    Window
    .partitionBy("lead_id")
    .orderBy(
        F.col("updated_date").desc_nulls_last()
    )
)

df_marketing_lead_clean = (
    df_marketing_lead_clean
    .withColumn(
        "_row_number",
        F.row_number().over(lead_window)
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)

print(
    "Records after deduplication:",
    df_marketing_lead_clean.count()
)

Records after deduplication: 20


In [0]:
# ============================================================
# MARKETING LEAD - SILVER PROCESSING METADATA
# ============================================================

# Record when the Silver transformation processed the lead.
df_marketing_lead_clean = (
    df_marketing_lead_clean
    .withColumn(
        "_silver_processed_timestamp",
        F.current_timestamp()
    )
)

display(df_marketing_lead_clean)

lead_id,customer_id,showroom_id,campaign_id,salesperson_id,lead_date,lead_source,lead_status,created_date,updated_date,_silver_processed_timestamp
1,1,1,2,1,2026-01-05,Google Ads,Converted,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z,2026-08-17T08:07:24.981Z
2,2,2,3,3,2026-01-12,Instagram,Qualified,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z,2026-08-17T08:07:24.981Z
3,3,3,1,5,2026-01-15,Walk-in,New,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z,2026-08-17T08:07:24.981Z
4,4,1,4,2,2026-01-18,Google Ads,Contacted,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z,2026-08-17T08:07:24.981Z
5,5,5,3,9,2026-01-22,Instagram,Qualified,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z,2026-08-17T08:07:24.981Z
6,6,2,5,4,2026-01-25,Website,Converted,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z,2026-08-17T08:07:24.981Z
7,7,4,6,7,2026-02-01,Facebook,Lost,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z,2026-08-17T08:07:24.981Z
8,8,5,7,10,2026-02-05,Referral,Contacted,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z,2026-08-17T08:07:24.981Z
9,9,7,8,13,2026-02-10,Google Ads,Qualified,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z,2026-08-17T08:07:24.981Z
10,10,3,9,6,2026-02-15,Website,Converted,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z,2026-08-17T08:07:24.981Z


In [0]:
# ============================================================
# MARKETING LEAD - WRITE TO SILVER DELTA
# ============================================================

# Save the cleaned marketing lead data as a Delta table.
#
# Catalog = showroom_analytics
# Schema  = silver
# Table   = marketing_lead

df_marketing_lead_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "showroom_analytics.silver.marketing_lead"
    )

In [0]:
# ============================================================
# MARKETING LEAD - VERIFY SILVER TABLE
# ============================================================

# Read the Silver Delta table
df_silver_marketing_lead = spark.table(
    "showroom_analytics.silver.marketing_lead"
)

# Display the transformed data
display(df_silver_marketing_lead)

# Display final schema
df_silver_marketing_lead.printSchema()

# Display final record count
print(
    "Silver marketing lead records:",
    df_silver_marketing_lead.count()
)

lead_id,customer_id,showroom_id,campaign_id,salesperson_id,lead_date,lead_source,lead_status,created_date,updated_date,_silver_processed_timestamp
1,1,1,2,1,2026-01-05,Google Ads,Converted,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z,2026-08-17T08:07:34.108Z
2,2,2,3,3,2026-01-12,Instagram,Qualified,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z,2026-08-17T08:07:34.108Z
3,3,3,1,5,2026-01-15,Walk-in,New,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z,2026-08-17T08:07:34.108Z
4,4,1,4,2,2026-01-18,Google Ads,Contacted,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z,2026-08-17T08:07:34.108Z
5,5,5,3,9,2026-01-22,Instagram,Qualified,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z,2026-08-17T08:07:34.108Z
6,6,2,5,4,2026-01-25,Website,Converted,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z,2026-08-17T08:07:34.108Z
7,7,4,6,7,2026-02-01,Facebook,Lost,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z,2026-08-17T08:07:34.108Z
8,8,5,7,10,2026-02-05,Referral,Contacted,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z,2026-08-17T08:07:34.108Z
9,9,7,8,13,2026-02-10,Google Ads,Qualified,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z,2026-08-17T08:07:34.108Z
10,10,3,9,6,2026-02-15,Website,Converted,2026-08-15T16:20:36.648Z,2026-08-15T16:20:36.648Z,2026-08-17T08:07:34.108Z


root
 |-- lead_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- showroom_id: integer (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- salesperson_id: integer (nullable = true)
 |-- lead_date: date (nullable = true)
 |-- lead_source: string (nullable = true)
 |-- lead_status: string (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- updated_date: timestamp (nullable = true)
 |-- _silver_processed_timestamp: timestamp (nullable = true)

Silver marketing lead records: 20


In [0]:
# ============================================================
# SHOWROOM VISIT - READ BRONZE DATA
# ============================================================

# Path to the showroom_visit Bronze folder
showroom_visit_path = (
    bronze_mssql_path + "showroom_visit/"
)

# Read Parquet files from Bronze
df_showroom_visit = spark.read.parquet(
    showroom_visit_path
)

# Display source data
display(df_showroom_visit)

# Display schema
df_showroom_visit.printSchema()

# Display record count
print(
    "Total showroom visit records:",
    df_showroom_visit.count()
)

visit_id,customer_id,showroom_id,salesperson_id,visit_date,visit_purpose,visit_status,created_date,updated_date
1,1,1,1,2026-01-08,New Car Inquiry,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
2,2,2,3,2026-01-12,Test Drive,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
3,3,3,5,2026-01-15,Price Inquiry,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
4,4,1,2,2026-01-18,New Car Inquiry,Scheduled,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
5,5,5,9,2026-01-22,Test Drive,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
6,6,2,4,2026-02-02,Exchange,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
7,7,4,7,2026-02-08,New Car Inquiry,Cancelled,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
8,8,5,10,2026-02-14,Test Drive,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
9,9,7,13,2026-02-20,Price Inquiry,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
10,10,3,6,2026-02-25,New Car Inquiry,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z


root
 |-- visit_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- showroom_id: integer (nullable = true)
 |-- salesperson_id: integer (nullable = true)
 |-- visit_date: date (nullable = true)
 |-- visit_purpose: string (nullable = true)
 |-- visit_status: string (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- updated_date: timestamp (nullable = true)

Total showroom visit records: 20


In [0]:
# ============================================================
# SHOWROOM VISIT - DATA QUALITY CHECK
# ============================================================

# Count NULL values in every column
df_showroom_visit.select([
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(f"{c}_nulls")
    for c in df_showroom_visit.columns
]).show()

# Check exact duplicate rows
total_records = df_showroom_visit.count()

distinct_records = (
    df_showroom_visit.dropDuplicates().count()
)

print(
    "Exact duplicate records:",
    total_records - distinct_records
)

+--------------+-----------------+-----------------+--------------------+----------------+-------------------+------------------+------------------+------------------+
|visit_id_nulls|customer_id_nulls|showroom_id_nulls|salesperson_id_nulls|visit_date_nulls|visit_purpose_nulls|visit_status_nulls|created_date_nulls|updated_date_nulls|
+--------------+-----------------+-----------------+--------------------+----------------+-------------------+------------------+------------------+------------------+
|             0|                0|                0|                   0|               0|                  0|                 0|                 0|                 0|
+--------------+-----------------+-----------------+--------------------+----------------+-------------------+------------------+------------------+------------------+

Exact duplicate records: 0


In [0]:
# ============================================================
# SHOWROOM VISIT - INSPECT SOURCE DATA
# ============================================================

# Display sample records
display(
    df_showroom_visit.limit(20)
)

visit_id,customer_id,showroom_id,salesperson_id,visit_date,visit_purpose,visit_status,created_date,updated_date
1,1,1,1,2026-01-08,New Car Inquiry,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
2,2,2,3,2026-01-12,Test Drive,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
3,3,3,5,2026-01-15,Price Inquiry,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
4,4,1,2,2026-01-18,New Car Inquiry,Scheduled,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
5,5,5,9,2026-01-22,Test Drive,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
6,6,2,4,2026-02-02,Exchange,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
7,7,4,7,2026-02-08,New Car Inquiry,Cancelled,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
8,8,5,10,2026-02-14,Test Drive,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
9,9,7,13,2026-02-20,Price Inquiry,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
10,10,3,6,2026-02-25,New Car Inquiry,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z


In [0]:
# ============================================================
# SHOWROOM VISIT - DATA QUALITY CHECK
# ============================================================

# Count NULL values in every column
df_showroom_visit.select([
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(f"{c}_nulls")
    for c in df_showroom_visit.columns
]).show()

# Check duplicate visit IDs
print("Duplicate visit IDs:")

df_showroom_visit.groupBy("visit_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

+--------------+-----------------+-----------------+--------------------+----------------+-------------------+------------------+------------------+------------------+
|visit_id_nulls|customer_id_nulls|showroom_id_nulls|salesperson_id_nulls|visit_date_nulls|visit_purpose_nulls|visit_status_nulls|created_date_nulls|updated_date_nulls|
+--------------+-----------------+-----------------+--------------------+----------------+-------------------+------------------+------------------+------------------+
|             0|                0|                0|                   0|               0|                  0|                 0|                 0|                 0|
+--------------+-----------------+-----------------+--------------------+----------------+-------------------+------------------+------------------+------------------+

Duplicate visit IDs:
+--------+-----+
|visit_id|count|
+--------+-----+
+--------+-----+



In [0]:
# ============================================================
# SHOWROOM VISIT - CLEAN STRING COLUMNS
# ============================================================

df_showroom_visit_clean = (
    df_showroom_visit

    # Remove extra spaces and standardize visit purpose
    .withColumn(
        "visit_purpose",
        F.initcap(
            F.trim(F.col("visit_purpose"))
        )
    )

    # Remove extra spaces and standardize visit status
    .withColumn(
        "visit_status",
        F.initcap(
            F.trim(F.col("visit_status"))
        )
    )
)

display(df_showroom_visit_clean)

visit_id,customer_id,showroom_id,salesperson_id,visit_date,visit_purpose,visit_status,created_date,updated_date
1,1,1,1,2026-01-08,New Car Inquiry,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
2,2,2,3,2026-01-12,Test Drive,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
3,3,3,5,2026-01-15,Price Inquiry,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
4,4,1,2,2026-01-18,New Car Inquiry,Scheduled,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
5,5,5,9,2026-01-22,Test Drive,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
6,6,2,4,2026-02-02,Exchange,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
7,7,4,7,2026-02-08,New Car Inquiry,Cancelled,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
8,8,5,10,2026-02-14,Test Drive,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
9,9,7,13,2026-02-20,Price Inquiry,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z
10,10,3,6,2026-02-25,New Car Inquiry,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z


In [0]:
# ============================================================
# SHOWROOM VISIT - VALIDATE RECORDS
# ============================================================

# visit_id is the business key and cannot be NULL
df_showroom_visit_clean = (
    df_showroom_visit_clean
    .filter(
        F.col("visit_id").isNotNull()
    )
)

print(
    "Records after visit ID validation:",
    df_showroom_visit_clean.count()
)

Records after visit ID validation: 20


In [0]:
# ============================================================
# SHOWROOM VISIT - REFERENTIAL INTEGRITY CHECK
# ============================================================

# Load existing Silver reference tables
df_customer_silver = spark.table(
    "showroom_analytics.silver.customer"
)

df_showroom_silver = spark.table(
    "showroom_analytics.silver.showroom"
)

df_salesperson_silver = spark.table(
    "showroom_analytics.silver.salesperson"
)

# ------------------------------------------------------------
# CUSTOMER CHECK
# ------------------------------------------------------------

invalid_customers = (
    df_showroom_visit_clean
    .join(
        df_customer_silver.select("customer_id"),
        "customer_id",
        "left_anti"
    )
)

print(
    "Invalid customer_id:",
    invalid_customers.count()
)

# ------------------------------------------------------------
# SHOWROOM CHECK
# ------------------------------------------------------------

invalid_showrooms = (
    df_showroom_visit_clean
    .join(
        df_showroom_silver.select("showroom_id"),
        "showroom_id",
        "left_anti"
    )
)

print(
    "Invalid showroom_id:",
    invalid_showrooms.count()
)

# ------------------------------------------------------------
# SALESPERSON CHECK
# ------------------------------------------------------------

invalid_salespersons = (
    df_showroom_visit_clean
    .join(
        df_salesperson_silver.select("salesperson_id"),
        "salesperson_id",
        "left_anti"
    )
)

print(
    "Invalid salesperson_id:",
    invalid_salespersons.count()
)

Invalid customer_id: 0
Invalid showroom_id: 0
Invalid salesperson_id: 0


In [0]:
# ============================================================
# SHOWROOM VISIT - CHECK STATUS VALUES
# ============================================================

df_showroom_visit_clean \
    .select("visit_status") \
    .distinct() \
    .show()

+------------+
|visit_status|
+------------+
|   Scheduled|
|   Cancelled|
|   Completed|
+------------+



In [0]:
# ============================================================
# SHOWROOM VISIT - DEDUPLICATION
# ============================================================

# Keep the latest version of every visit
visit_window = (
    Window
    .partitionBy("visit_id")
    .orderBy(
        F.col("updated_date").desc_nulls_last()
    )
)

df_showroom_visit_clean = (
    df_showroom_visit_clean
    .withColumn(
        "_row_number",
        F.row_number().over(visit_window)
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)

print(
    "Records after deduplication:",
    df_showroom_visit_clean.count()
)

Records after deduplication: 20


In [0]:
# ============================================================
# SHOWROOM VISIT - SILVER METADATA
# ============================================================

df_showroom_visit_clean = (
    df_showroom_visit_clean
    .withColumn(
        "_silver_processed_timestamp",
        F.current_timestamp()
    )
)

display(df_showroom_visit_clean)

visit_id,customer_id,showroom_id,salesperson_id,visit_date,visit_purpose,visit_status,created_date,updated_date,_silver_processed_timestamp
1,1,1,1,2026-01-08,New Car Inquiry,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z,2026-08-17T08:12:24.613Z
2,2,2,3,2026-01-12,Test Drive,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z,2026-08-17T08:12:24.613Z
3,3,3,5,2026-01-15,Price Inquiry,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z,2026-08-17T08:12:24.613Z
4,4,1,2,2026-01-18,New Car Inquiry,Scheduled,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z,2026-08-17T08:12:24.613Z
5,5,5,9,2026-01-22,Test Drive,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z,2026-08-17T08:12:24.613Z
6,6,2,4,2026-02-02,Exchange,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z,2026-08-17T08:12:24.613Z
7,7,4,7,2026-02-08,New Car Inquiry,Cancelled,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z,2026-08-17T08:12:24.613Z
8,8,5,10,2026-02-14,Test Drive,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z,2026-08-17T08:12:24.613Z
9,9,7,13,2026-02-20,Price Inquiry,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z,2026-08-17T08:12:24.613Z
10,10,3,6,2026-02-25,New Car Inquiry,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z,2026-08-17T08:12:24.613Z


In [0]:
# ============================================================
# SHOWROOM VISIT - WRITE TO SILVER DELTA TABLE
# ============================================================

# Write the transformed DataFrame into the Silver layer.
# Existing table will be replaced during this development phase.

df_showroom_visit_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "showroom_analytics.silver.showroom_visit"
    )

print("SUCCESS: showroom_visit written to Silver.")

SUCCESS: showroom_visit written to Silver.


In [0]:
# ============================================================
# SHOWROOM VISIT - VERIFY SILVER TABLE
# ============================================================

df_silver_showroom_visit = spark.table(
    "showroom_analytics.silver.showroom_visit"
)

display(df_silver_showroom_visit)

df_silver_showroom_visit.printSchema()

print(
    "Silver showroom_visit records:",
    df_silver_showroom_visit.count()
)

visit_id,customer_id,showroom_id,salesperson_id,visit_date,visit_purpose,visit_status,created_date,updated_date,_silver_processed_timestamp
1,1,1,1,2026-01-08,New Car Inquiry,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z,2026-08-17T08:13:52.495Z
2,2,2,3,2026-01-12,Test Drive,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z,2026-08-17T08:13:52.495Z
3,3,3,5,2026-01-15,Price Inquiry,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z,2026-08-17T08:13:52.495Z
4,4,1,2,2026-01-18,New Car Inquiry,Scheduled,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z,2026-08-17T08:13:52.495Z
5,5,5,9,2026-01-22,Test Drive,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z,2026-08-17T08:13:52.495Z
6,6,2,4,2026-02-02,Exchange,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z,2026-08-17T08:13:52.495Z
7,7,4,7,2026-02-08,New Car Inquiry,Cancelled,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z,2026-08-17T08:13:52.495Z
8,8,5,10,2026-02-14,Test Drive,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z,2026-08-17T08:13:52.495Z
9,9,7,13,2026-02-20,Price Inquiry,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z,2026-08-17T08:13:52.495Z
10,10,3,6,2026-02-25,New Car Inquiry,Completed,2026-08-15T16:15:42.635Z,2026-08-15T16:15:42.635Z,2026-08-17T08:13:52.495Z


root
 |-- visit_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- showroom_id: integer (nullable = true)
 |-- salesperson_id: integer (nullable = true)
 |-- visit_date: date (nullable = true)
 |-- visit_purpose: string (nullable = true)
 |-- visit_status: string (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- updated_date: timestamp (nullable = true)
 |-- _silver_processed_timestamp: timestamp (nullable = true)

Silver showroom_visit records: 20


In [0]:
# ============================================================
# TEST DRIVE - READ BRONZE DATA
# ============================================================

test_drive_path = (
    bronze_mssql_path + "test_drive/"
)

df_test_drive = spark.read.parquet(
    test_drive_path
)

display(df_test_drive)

df_test_drive.printSchema()

print(
    "Total test drive records:",
    df_test_drive.count()
)

test_drive_id,customer_id,vehicle_id,showroom_id,salesperson_id,test_drive_date,test_drive_status,created_date,updated_date
1,1,10001,1,1,2026-01-09,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
2,2,10008,2,3,2026-01-13,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
3,3,10016,3,5,2026-01-16,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
4,4,10002,1,2,2026-01-19,No Show,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
5,5,10027,5,9,2026-01-23,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
6,6,10009,2,4,2026-02-03,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
7,7,10023,4,7,2026-02-09,Cancelled,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
8,8,10028,5,10,2026-02-15,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
9,9,10034,7,13,2026-02-21,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
10,10,10005,3,6,2026-02-26,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z


root
 |-- test_drive_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- vehicle_id: integer (nullable = true)
 |-- showroom_id: integer (nullable = true)
 |-- salesperson_id: integer (nullable = true)
 |-- test_drive_date: date (nullable = true)
 |-- test_drive_status: string (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- updated_date: timestamp (nullable = true)

Total test drive records: 20


In [0]:
# ============================================================
# TEST DRIVE - DATA QUALITY CHECK
# ============================================================

# NULL count for every column
df_test_drive.select([
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(f"{c}_nulls")
    for c in df_test_drive.columns
]).show()

# Check duplicate rows based on the complete record
print("Exact duplicate records:")

total_records = df_test_drive.count()

distinct_records = (
    df_test_drive.dropDuplicates().count()
)

print(
    total_records - distinct_records
)

+-------------------+-----------------+----------------+-----------------+--------------------+---------------------+-----------------------+------------------+------------------+
|test_drive_id_nulls|customer_id_nulls|vehicle_id_nulls|showroom_id_nulls|salesperson_id_nulls|test_drive_date_nulls|test_drive_status_nulls|created_date_nulls|updated_date_nulls|
+-------------------+-----------------+----------------+-----------------+--------------------+---------------------+-----------------------+------------------+------------------+
|                  0|                0|               0|                0|                   0|                    0|                      0|                 0|                 0|
+-------------------+-----------------+----------------+-----------------+--------------------+---------------------+-----------------------+------------------+------------------+

Exact duplicate records:
0


In [0]:
# ============================================================
# TEST DRIVE - INSPECT SOURCE DATA
# ============================================================

display(
    df_test_drive.limit(20)
)

test_drive_id,customer_id,vehicle_id,showroom_id,salesperson_id,test_drive_date,test_drive_status,created_date,updated_date
1,1,10001,1,1,2026-01-09,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
2,2,10008,2,3,2026-01-13,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
3,3,10016,3,5,2026-01-16,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
4,4,10002,1,2,2026-01-19,No Show,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
5,5,10027,5,9,2026-01-23,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
6,6,10009,2,4,2026-02-03,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
7,7,10023,4,7,2026-02-09,Cancelled,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
8,8,10028,5,10,2026-02-15,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
9,9,10034,7,13,2026-02-21,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
10,10,10005,3,6,2026-02-26,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z


In [0]:
# ============================================================
# TEST DRIVE - CLEAN STRING COLUMNS
# ============================================================

df_test_drive_clean = (
    df_test_drive

    # Remove unnecessary spaces and standardize status
    .withColumn(
        "test_drive_status",
        F.initcap(
            F.trim(
                F.col("test_drive_status")
            )
        )
    )
)

display(df_test_drive_clean)

test_drive_id,customer_id,vehicle_id,showroom_id,salesperson_id,test_drive_date,test_drive_status,created_date,updated_date
1,1,10001,1,1,2026-01-09,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
2,2,10008,2,3,2026-01-13,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
3,3,10016,3,5,2026-01-16,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
4,4,10002,1,2,2026-01-19,No Show,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
5,5,10027,5,9,2026-01-23,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
6,6,10009,2,4,2026-02-03,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
7,7,10023,4,7,2026-02-09,Cancelled,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
8,8,10028,5,10,2026-02-15,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
9,9,10034,7,13,2026-02-21,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z
10,10,10005,3,6,2026-02-26,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z


In [0]:
# ============================================================
# TEST DRIVE - VALIDATE RECORDS
# ============================================================

# test_drive_id is the business key and must be present.
df_test_drive_clean = (
    df_test_drive_clean
    .filter(
        F.col("test_drive_id").isNotNull()
    )
)

print(
    "Records after test_drive ID validation:",
    df_test_drive_clean.count()
)

Records after test_drive ID validation: 20


In [0]:
# ============================================================
# TEST DRIVE - REFERENTIAL INTEGRITY CHECK
# ============================================================

# Load existing Silver reference tables
df_customer_silver = spark.table(
    "showroom_analytics.silver.customer"
)

df_showroom_silver = spark.table(
    "showroom_analytics.silver.showroom"
)

df_salesperson_silver = spark.table(
    "showroom_analytics.silver.salesperson"
)

# ------------------------------------------------------------
# CUSTOMER CHECK
# ------------------------------------------------------------

invalid_customers = (
    df_test_drive_clean
    .join(
        df_customer_silver.select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
)

print(
    "Invalid customer_id:",
    invalid_customers.count()
)

# ------------------------------------------------------------
# SHOWROOM CHECK
# ------------------------------------------------------------

invalid_showrooms = (
    df_test_drive_clean
    .join(
        df_showroom_silver.select("showroom_id"),
        on="showroom_id",
        how="left_anti"
    )
)

print(
    "Invalid showroom_id:",
    invalid_showrooms.count()
)

# ------------------------------------------------------------
# SALESPERSON CHECK
# ------------------------------------------------------------

invalid_salespersons = (
    df_test_drive_clean
    .join(
        df_salesperson_silver.select("salesperson_id"),
        on="salesperson_id",
        how="left_anti"
    )
)

print(
    "Invalid salesperson_id:",
    invalid_salespersons.count()
)

Invalid customer_id: 0
Invalid showroom_id: 0
Invalid salesperson_id: 0


In [0]:
# ============================================================
# TEST DRIVE - VEHICLE ID VALIDATION
# ============================================================

# vehicle_id will be checked against the Silver vehicle table
# after we transform the vehicle table.
#
# For now, make sure the source contains a vehicle ID.

missing_vehicle_ids = (
    df_test_drive_clean
    .filter(
        F.col("vehicle_id").isNull()
    )
)

print(
    "Test drives with missing vehicle_id:",
    missing_vehicle_ids.count()
)

Test drives with missing vehicle_id: 0


In [0]:
# ============================================================
# TEST DRIVE - CHECK STATUS VALUES
# ============================================================

print("Distinct test drive statuses:")

df_test_drive_clean \
    .select("test_drive_status") \
    .distinct() \
    .show()

Distinct test drive statuses:
+-----------------+
|test_drive_status|
+-----------------+
|          No Show|
|        Cancelled|
|        Completed|
+-----------------+



In [0]:
# ============================================================
# TEST DRIVE - DEDUPLICATION
# ============================================================

# Keep the latest version of every test drive.
# updated_date determines the newest record.

test_drive_window = (
    Window
    .partitionBy("test_drive_id")
    .orderBy(
        F.col("updated_date").desc_nulls_last()
    )
)

df_test_drive_clean = (
    df_test_drive_clean
    .withColumn(
        "_row_number",
        F.row_number().over(test_drive_window)
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)

print(
    "Records after deduplication:",
    df_test_drive_clean.count()
)

Records after deduplication: 20


In [0]:
# ============================================================
# TEST DRIVE - SILVER PROCESSING METADATA
# ============================================================

df_test_drive_clean = (
    df_test_drive_clean
    .withColumn(
        "_silver_processed_timestamp",
        F.current_timestamp()
    )
)

display(df_test_drive_clean)

test_drive_id,customer_id,vehicle_id,showroom_id,salesperson_id,test_drive_date,test_drive_status,created_date,updated_date,_silver_processed_timestamp
1,1,10001,1,1,2026-01-09,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z,2026-08-17T08:22:46.218Z
2,2,10008,2,3,2026-01-13,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z,2026-08-17T08:22:46.218Z
3,3,10016,3,5,2026-01-16,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z,2026-08-17T08:22:46.218Z
4,4,10002,1,2,2026-01-19,No Show,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z,2026-08-17T08:22:46.218Z
5,5,10027,5,9,2026-01-23,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z,2026-08-17T08:22:46.218Z
6,6,10009,2,4,2026-02-03,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z,2026-08-17T08:22:46.218Z
7,7,10023,4,7,2026-02-09,Cancelled,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z,2026-08-17T08:22:46.218Z
8,8,10028,5,10,2026-02-15,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z,2026-08-17T08:22:46.218Z
9,9,10034,7,13,2026-02-21,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z,2026-08-17T08:22:46.218Z
10,10,10005,3,6,2026-02-26,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z,2026-08-17T08:22:46.218Z


In [0]:
# ============================================================
# TEST DRIVE - WRITE TO SILVER DELTA
# ============================================================

# Save the transformed test drive data.
#
# Catalog = showroom_analytics
# Schema  = silver
# Table   = test_drive

df_test_drive_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "showroom_analytics.silver.test_drive"
    )

print(
    "SUCCESS: test_drive written to Silver."
)

SUCCESS: test_drive written to Silver.


In [0]:
# ============================================================
# TEST DRIVE - VERIFY SILVER TABLE
# ============================================================

df_silver_test_drive = spark.table(
    "showroom_analytics.silver.test_drive"
)

display(df_silver_test_drive)

df_silver_test_drive.printSchema()

print(
    "Silver test_drive records:",
    df_silver_test_drive.count()
)

test_drive_id,customer_id,vehicle_id,showroom_id,salesperson_id,test_drive_date,test_drive_status,created_date,updated_date,_silver_processed_timestamp
1,1,10001,1,1,2026-01-09,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z,2026-08-17T08:22:56.920Z
2,2,10008,2,3,2026-01-13,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z,2026-08-17T08:22:56.920Z
3,3,10016,3,5,2026-01-16,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z,2026-08-17T08:22:56.920Z
4,4,10002,1,2,2026-01-19,No Show,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z,2026-08-17T08:22:56.920Z
5,5,10027,5,9,2026-01-23,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z,2026-08-17T08:22:56.920Z
6,6,10009,2,4,2026-02-03,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z,2026-08-17T08:22:56.920Z
7,7,10023,4,7,2026-02-09,Cancelled,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z,2026-08-17T08:22:56.920Z
8,8,10028,5,10,2026-02-15,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z,2026-08-17T08:22:56.920Z
9,9,10034,7,13,2026-02-21,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z,2026-08-17T08:22:56.920Z
10,10,10005,3,6,2026-02-26,Completed,2026-08-15T16:15:42.669Z,2026-08-15T16:15:42.669Z,2026-08-17T08:22:56.920Z


root
 |-- test_drive_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- vehicle_id: integer (nullable = true)
 |-- showroom_id: integer (nullable = true)
 |-- salesperson_id: integer (nullable = true)
 |-- test_drive_date: date (nullable = true)
 |-- test_drive_status: string (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- updated_date: timestamp (nullable = true)
 |-- _silver_processed_timestamp: timestamp (nullable = true)

Silver test_drive records: 20


In [0]:
# ============================================================
# CUSTOMER FOLLOWUP - READ BRONZE DATA
# ============================================================

# Path to the customer_followup Bronze folder
customer_followup_path = (
    bronze_mssql_path + "customer_followup/"
)

# Read Parquet data from Bronze
df_customer_followup = spark.read.parquet(
    customer_followup_path
)

# Display source records
display(df_customer_followup)

# Display schema
df_customer_followup.printSchema()

# Display record count
print(
    "Total customer followup records:",
    df_customer_followup.count()
)

followup_id,lead_id,customer_id,salesperson_id,showroom_id,followup_date,followup_type,followup_status,remarks,created_date,updated_date
1,1,1,1,1,2026-01-10,Phone,Completed,Customer interested in Nexon,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z
2,2,2,3,2,2026-01-14,WhatsApp,Completed,Customer requested price details,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z
3,3,3,5,3,2026-01-17,Phone,Pending,Customer considering multiple models,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z
4,4,4,2,1,2026-01-20,Email,Completed,Quotation sent to customer,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z
5,5,5,9,5,2026-01-24,WhatsApp,Completed,Customer interested in Seltos,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z
6,6,6,4,2,2026-02-04,Phone,Completed,Exchange details discussed,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z
7,7,7,7,4,2026-02-10,Email,No Response,Customer did not respond,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z
8,8,8,10,5,2026-02-16,Phone,Completed,Test drive feedback received,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z
9,9,9,13,7,2026-02-22,WhatsApp,Pending,Customer asked for finance options,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z
10,10,10,6,3,2026-02-27,Phone,Completed,Customer ready for booking,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z


root
 |-- followup_id: long (nullable = true)
 |-- lead_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- salesperson_id: integer (nullable = true)
 |-- showroom_id: integer (nullable = true)
 |-- followup_date: date (nullable = true)
 |-- followup_type: string (nullable = true)
 |-- followup_status: string (nullable = true)
 |-- remarks: string (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- updated_date: timestamp (nullable = true)

Total customer followup records: 20


In [0]:
# ============================================================
# CUSTOMER FOLLOWUP - CLEAN AND VALIDATE
# ============================================================

# Standardize text fields by removing extra spaces
# and applying consistent capitalization.
df_customer_followup_clean = (
    df_customer_followup
    .withColumn(
        "followup_type",
        F.initcap(F.trim(F.col("followup_type")))
    )
    .withColumn(
        "followup_status",
        F.initcap(F.trim(F.col("followup_status")))
    )
    .withColumn(
        "remarks",
        F.trim(F.col("remarks"))
    )
    # followup_id is the business key and must be present
    .filter(
        F.col("followup_id").isNotNull()
    )
)

display(df_customer_followup_clean)

followup_id,lead_id,customer_id,salesperson_id,showroom_id,followup_date,followup_type,followup_status,remarks,created_date,updated_date
1,1,1,1,1,2026-01-10,Phone,Completed,Customer interested in Nexon,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z
2,2,2,3,2,2026-01-14,Whatsapp,Completed,Customer requested price details,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z
3,3,3,5,3,2026-01-17,Phone,Pending,Customer considering multiple models,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z
4,4,4,2,1,2026-01-20,Email,Completed,Quotation sent to customer,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z
5,5,5,9,5,2026-01-24,Whatsapp,Completed,Customer interested in Seltos,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z
6,6,6,4,2,2026-02-04,Phone,Completed,Exchange details discussed,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z
7,7,7,7,4,2026-02-10,Email,No Response,Customer did not respond,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z
8,8,8,10,5,2026-02-16,Phone,Completed,Test drive feedback received,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z
9,9,9,13,7,2026-02-22,Whatsapp,Pending,Customer asked for finance options,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z
10,10,10,6,3,2026-02-27,Phone,Completed,Customer ready for booking,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z


In [0]:
# ============================================================
# CUSTOMER FOLLOWUP - REFERENTIAL INTEGRITY CHECK
# ============================================================

# Check lead references
invalid_leads = (
    df_customer_followup_clean
    .join(
        spark.table(
            "showroom_analytics.silver.marketing_lead"
        ).select("lead_id"),
        "lead_id",
        "left_anti"
    )
)

# Check customer references
invalid_customers = (
    df_customer_followup_clean
    .join(
        spark.table(
            "showroom_analytics.silver.customer"
        ).select("customer_id"),
        "customer_id",
        "left_anti"
    )
)

# Check salesperson references
invalid_salespersons = (
    df_customer_followup_clean
    .join(
        spark.table(
            "showroom_analytics.silver.salesperson"
        ).select("salesperson_id"),
        "salesperson_id",
        "left_anti"
    )
)

# Check showroom references
invalid_showrooms = (
    df_customer_followup_clean
    .join(
        spark.table(
            "showroom_analytics.silver.showroom"
        ).select("showroom_id"),
        "showroom_id",
        "left_anti"
    )
)

print("Invalid lead_id:", invalid_leads.count())
print("Invalid customer_id:", invalid_customers.count())
print("Invalid salesperson_id:", invalid_salespersons.count())
print("Invalid showroom_id:", invalid_showrooms.count())

Invalid lead_id: 0
Invalid customer_id: 0
Invalid salesperson_id: 0
Invalid showroom_id: 0


In [0]:
# ============================================================
# CUSTOMER FOLLOWUP - DEDUPLICATION
# ============================================================

# Keep the latest version of each follow-up
# using updated_date.
followup_window = (
    Window
    .partitionBy("followup_id")
    .orderBy(
        F.col("updated_date").desc_nulls_last()
    )
)

df_customer_followup_clean = (
    df_customer_followup_clean
    .withColumn(
        "_row_number",
        F.row_number().over(followup_window)
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)

In [0]:
# ============================================================
# CUSTOMER FOLLOWUP - SILVER METADATA
# ============================================================

# Store the time when Silver processed the record.
df_customer_followup_clean = (
    df_customer_followup_clean
    .withColumn(
        "_silver_processed_timestamp",
        F.current_timestamp()
    )
)

In [0]:
# ============================================================
# CUSTOMER FOLLOWUP - WRITE TO SILVER
# ============================================================

df_customer_followup_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "showroom_analytics.silver.customer_followup"
    )

print(
    "SUCCESS: customer_followup written to Silver."
)

SUCCESS: customer_followup written to Silver.


In [0]:
# ============================================================
# CUSTOMER FOLLOWUP - VERIFY SILVER
# ============================================================

df_silver_customer_followup = spark.table(
    "showroom_analytics.silver.customer_followup"
)

display(df_silver_customer_followup)

print(
    "Silver customer_followup records:",
    df_silver_customer_followup.count()
)

followup_id,lead_id,customer_id,salesperson_id,showroom_id,followup_date,followup_type,followup_status,remarks,created_date,updated_date,_silver_processed_timestamp
1,1,1,1,1,2026-01-10,Phone,Completed,Customer interested in Nexon,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z,2026-08-17T08:26:06.841Z
2,2,2,3,2,2026-01-14,Whatsapp,Completed,Customer requested price details,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z,2026-08-17T08:26:06.841Z
3,3,3,5,3,2026-01-17,Phone,Pending,Customer considering multiple models,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z,2026-08-17T08:26:06.841Z
4,4,4,2,1,2026-01-20,Email,Completed,Quotation sent to customer,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z,2026-08-17T08:26:06.841Z
5,5,5,9,5,2026-01-24,Whatsapp,Completed,Customer interested in Seltos,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z,2026-08-17T08:26:06.841Z
6,6,6,4,2,2026-02-04,Phone,Completed,Exchange details discussed,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z,2026-08-17T08:26:06.841Z
7,7,7,7,4,2026-02-10,Email,No Response,Customer did not respond,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z,2026-08-17T08:26:06.841Z
8,8,8,10,5,2026-02-16,Phone,Completed,Test drive feedback received,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z,2026-08-17T08:26:06.841Z
9,9,9,13,7,2026-02-22,Whatsapp,Pending,Customer asked for finance options,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z,2026-08-17T08:26:06.841Z
10,10,10,6,3,2026-02-27,Phone,Completed,Customer ready for booking,2026-08-15T16:21:27.548Z,2026-08-15T16:21:27.548Z,2026-08-17T08:26:06.841Z


Silver customer_followup records: 20


In [0]:
# ============================================================
# INVENTORY - READ BRONZE DATA
# ============================================================

# Path to the inventory Bronze folder
inventory_path = (
    bronze_mssql_path + "inventory/"
)

# Read inventory Parquet data
df_inventory = spark.read.parquet(
    inventory_path
)

# Display source data
display(df_inventory)

# Display schema
df_inventory.printSchema()

# Display record count
print(
    "Total inventory records:",
    df_inventory.count()
)

inventory_id,vehicle_id,showroom_id,vin,purchase_date,purchase_cost,status,created_date,updated_date
1,10001,1,VIN20260001,2026-01-05,1050000.00,Sold,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z
2,10002,1,VIN20260002,2026-01-12,1200000.00,Sold,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z
3,10008,2,VIN20260003,2026-01-18,1500000.00,Sold,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z
4,10009,2,VIN20260004,2026-02-01,1600000.00,Available,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z
5,10016,3,VIN20260005,2026-02-05,720000.00,Sold,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z
6,10017,3,VIN20260006,2026-02-10,850000.00,Available,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z
7,10023,4,VIN20260007,2026-02-15,1200000.00,Sold,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z
8,10024,4,VIN20260008,2026-02-20,1650000.00,Reserved,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z
9,10027,5,VIN20260009,2026-02-25,1600000.00,Sold,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z
10,10028,5,VIN20260010,2026-03-01,1250000.00,Available,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z


root
 |-- inventory_id: long (nullable = true)
 |-- vehicle_id: integer (nullable = true)
 |-- showroom_id: integer (nullable = true)
 |-- vin: string (nullable = true)
 |-- purchase_date: date (nullable = true)
 |-- purchase_cost: decimal(18,2) (nullable = true)
 |-- status: string (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- updated_date: timestamp (nullable = true)

Total inventory records: 20


In [0]:
# ============================================================
# INVENTORY - CLEAN AND HANDLE NULLS
# ============================================================

# Clean text fields and remove records where required
# business columns are NULL.

df_inventory_clean = (
    df_inventory

    # Remove unnecessary spaces from VIN
    .withColumn(
        "vin",
        F.upper(F.trim(F.col("vin")))
    )

    # Standardize inventory status
    .withColumn(
        "status",
        F.initcap(F.trim(F.col("status")))
    )

    # Required columns:
    # inventory_id, vehicle_id, showroom_id and VIN
    # must be available for a valid inventory record.
    .filter(
        F.col("inventory_id").isNotNull()
        & F.col("vehicle_id").isNotNull()
        & F.col("showroom_id").isNotNull()
        & F.col("vin").isNotNull()
    )
)

print(
    "Records after required-field NULL removal:",
    df_inventory_clean.count()
)

display(df_inventory_clean)

Records after required-field NULL removal: 20


inventory_id,vehicle_id,showroom_id,vin,purchase_date,purchase_cost,status,created_date,updated_date
1,10001,1,VIN20260001,2026-01-05,1050000.00,Sold,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z
2,10002,1,VIN20260002,2026-01-12,1200000.00,Sold,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z
3,10008,2,VIN20260003,2026-01-18,1500000.00,Sold,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z
4,10009,2,VIN20260004,2026-02-01,1600000.00,Available,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z
5,10016,3,VIN20260005,2026-02-05,720000.00,Sold,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z
6,10017,3,VIN20260006,2026-02-10,850000.00,Available,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z
7,10023,4,VIN20260007,2026-02-15,1200000.00,Sold,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z
8,10024,4,VIN20260008,2026-02-20,1650000.00,Reserved,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z
9,10027,5,VIN20260009,2026-02-25,1600000.00,Sold,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z
10,10028,5,VIN20260010,2026-03-01,1250000.00,Available,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z


In [0]:
# ============================================================
# INVENTORY - BUSINESS VALIDATION
# ============================================================

# Check for duplicate inventory IDs
print("Duplicate inventory IDs:")

df_inventory_clean.groupBy("inventory_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

# Check for duplicate VINs
print("Duplicate VINs:")

df_inventory_clean.groupBy("vin") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

# Check for negative purchase costs
print("Negative purchase costs:")

df_inventory_clean.filter(
    F.col("purchase_cost") < 0
).show()

# Check allowed inventory statuses
print("Inventory statuses:")

df_inventory_clean \
    .select("status") \
    .distinct() \
    .show()

Duplicate inventory IDs:
+------------+-----+
|inventory_id|count|
+------------+-----+
+------------+-----+

Duplicate VINs:
+---+-----+
|vin|count|
+---+-----+
+---+-----+

Negative purchase costs:
+------------+----------+-----------+---+-------------+-------------+------+------------+------------+
|inventory_id|vehicle_id|showroom_id|vin|purchase_date|purchase_cost|status|created_date|updated_date|
+------------+----------+-----------+---+-------------+-------------+------+------------+------------+
+------------+----------+-----------+---+-------------+-------------+------+------------+------------+

Inventory statuses:
+---------+
|   status|
+---------+
|     Sold|
|Available|
| Reserved|
+---------+



In [0]:
# ============================================================
# INVENTORY - SHOWROOM REFERENTIAL INTEGRITY
# ============================================================

# Find inventory records whose showroom_id
# does not exist in the Silver showroom table.

df_showroom_silver = spark.table(
    "showroom_analytics.silver.showroom"
)

invalid_showrooms = (
    df_inventory_clean
    .join(
        df_showroom_silver.select("showroom_id"),
        on="showroom_id",
        how="left_anti"
    )
)

print(
    "Inventory records with invalid showroom_id:",
    invalid_showrooms.count()
)

display(invalid_showrooms)

Inventory records with invalid showroom_id: 0


showroom_id,inventory_id,vehicle_id,vin,purchase_date,purchase_cost,status,created_date,updated_date


In [0]:
# ============================================================
# INVENTORY - DEDUPLICATION
# ============================================================

# Keep the latest version of each inventory record
# based on inventory_id and updated_date.

inventory_window = (
    Window
    .partitionBy("inventory_id")
    .orderBy(
        F.col("updated_date").desc_nulls_last()
    )
)

df_inventory_clean = (
    df_inventory_clean
    .withColumn(
        "_row_number",
        F.row_number().over(inventory_window)
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)

print(
    "Records after deduplication:",
    df_inventory_clean.count()
)

Records after deduplication: 20


In [0]:
# ============================================================
# INVENTORY - SILVER PROCESSING METADATA
# ============================================================

# Store the timestamp when the Silver layer processed
# the inventory record.

df_inventory_clean = (
    df_inventory_clean
    .withColumn(
        "_silver_processed_timestamp",
        F.current_timestamp()
    )
)

display(df_inventory_clean)

inventory_id,vehicle_id,showroom_id,vin,purchase_date,purchase_cost,status,created_date,updated_date,_silver_processed_timestamp
1,10001,1,VIN20260001,2026-01-05,1050000.00,Sold,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z,2026-08-17T08:28:11.071Z
2,10002,1,VIN20260002,2026-01-12,1200000.00,Sold,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z,2026-08-17T08:28:11.071Z
3,10008,2,VIN20260003,2026-01-18,1500000.00,Sold,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z,2026-08-17T08:28:11.071Z
4,10009,2,VIN20260004,2026-02-01,1600000.00,Available,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z,2026-08-17T08:28:11.071Z
5,10016,3,VIN20260005,2026-02-05,720000.00,Sold,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z,2026-08-17T08:28:11.071Z
6,10017,3,VIN20260006,2026-02-10,850000.00,Available,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z,2026-08-17T08:28:11.071Z
7,10023,4,VIN20260007,2026-02-15,1200000.00,Sold,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z,2026-08-17T08:28:11.071Z
8,10024,4,VIN20260008,2026-02-20,1650000.00,Reserved,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z,2026-08-17T08:28:11.071Z
9,10027,5,VIN20260009,2026-02-25,1600000.00,Sold,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z,2026-08-17T08:28:11.071Z
10,10028,5,VIN20260010,2026-03-01,1250000.00,Available,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z,2026-08-17T08:28:11.071Z


In [0]:
# ============================================================
# INVENTORY - WRITE TO SILVER DELTA
# ============================================================

df_inventory_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "showroom_analytics.silver.inventory"
    )

print(
    "SUCCESS: inventory written to Silver."
)

SUCCESS: inventory written to Silver.


In [0]:
# ============================================================
# INVENTORY - VERIFY SILVER
# ============================================================

df_silver_inventory = spark.table(
    "showroom_analytics.silver.inventory"
)

display(df_silver_inventory)

print(
    "Silver inventory records:",
    df_silver_inventory.count()
)

df_silver_inventory.printSchema()

inventory_id,vehicle_id,showroom_id,vin,purchase_date,purchase_cost,status,created_date,updated_date,_silver_processed_timestamp
1,10001,1,VIN20260001,2026-01-05,1050000.00,Sold,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z,2026-08-17T08:28:21.571Z
2,10002,1,VIN20260002,2026-01-12,1200000.00,Sold,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z,2026-08-17T08:28:21.571Z
3,10008,2,VIN20260003,2026-01-18,1500000.00,Sold,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z,2026-08-17T08:28:21.571Z
4,10009,2,VIN20260004,2026-02-01,1600000.00,Available,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z,2026-08-17T08:28:21.571Z
5,10016,3,VIN20260005,2026-02-05,720000.00,Sold,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z,2026-08-17T08:28:21.571Z
6,10017,3,VIN20260006,2026-02-10,850000.00,Available,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z,2026-08-17T08:28:21.571Z
7,10023,4,VIN20260007,2026-02-15,1200000.00,Sold,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z,2026-08-17T08:28:21.571Z
8,10024,4,VIN20260008,2026-02-20,1650000.00,Reserved,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z,2026-08-17T08:28:21.571Z
9,10027,5,VIN20260009,2026-02-25,1600000.00,Sold,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z,2026-08-17T08:28:21.571Z
10,10028,5,VIN20260010,2026-03-01,1250000.00,Available,2026-08-15T16:15:42.742Z,2026-08-15T16:15:42.742Z,2026-08-17T08:28:21.571Z


Silver inventory records: 20
root
 |-- inventory_id: long (nullable = true)
 |-- vehicle_id: integer (nullable = true)
 |-- showroom_id: integer (nullable = true)
 |-- vin: string (nullable = true)
 |-- purchase_date: date (nullable = true)
 |-- purchase_cost: decimal(18,2) (nullable = true)
 |-- status: string (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- updated_date: timestamp (nullable = true)
 |-- _silver_processed_timestamp: timestamp (nullable = true)



In [0]:
# ============================================================
# SALES - READ BRONZE DATA
# ============================================================

# Path to the sales Bronze folder
sales_path = (
    bronze_mssql_path + "sales/"
)

# Read sales Parquet data from Bronze
df_sales = spark.read.parquet(
    sales_path
)

# Display source data
display(df_sales)

# Display schema
df_sales.printSchema()

# Display record count
print(
    "Total sales records:",
    df_sales.count()
)

sale_id,customer_id,vehicle_id,showroom_id,salesperson_id,inventory_id,sale_date,sale_price,purchase_cost,discount,commission,payment_method,created_date,updated_date
1,1,10001,1,1,1,2026-01-10,1180000.00,1050000.00,20000.00,15000.00,Loan,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
2,2,10002,1,2,2,2026-01-20,1350000.00,1200000.00,30000.00,18000.00,Bank Transfer,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
3,3,10008,2,3,3,2026-01-25,1680000.00,1500000.00,25000.00,20000.00,Loan,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
4,5,10016,3,5,5,2026-02-12,830000.00,720000.00,10000.00,9000.00,UPI,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
5,7,10023,4,7,7,2026-02-22,1420000.00,1200000.00,30000.00,16000.00,Loan,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
6,8,10027,5,9,9,2026-03-02,1760000.00,1600000.00,20000.00,18000.00,Bank Transfer,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
7,10,10030,6,11,11,2026-03-12,2450000.00,2200000.00,50000.00,25000.00,Loan,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
8,13,10034,7,13,13,2026-03-25,1690000.00,1550000.00,25000.00,17000.00,UPI,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
9,15,10037,9,15,15,2026-04-05,1880000.00,1700000.00,30000.00,19000.00,Loan,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
10,18,10005,3,6,18,2026-04-18,1080000.00,950000.00,15000.00,10000.00,Cash,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z


root
 |-- sale_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- vehicle_id: integer (nullable = true)
 |-- showroom_id: integer (nullable = true)
 |-- salesperson_id: integer (nullable = true)
 |-- inventory_id: long (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- sale_price: decimal(18,2) (nullable = true)
 |-- purchase_cost: decimal(18,2) (nullable = true)
 |-- discount: decimal(18,2) (nullable = true)
 |-- commission: decimal(18,2) (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- updated_date: timestamp (nullable = true)

Total sales records: 11


In [0]:
# ============================================================
# SALES - INSPECT SOURCE DATA
# ============================================================

# Display sample sales records
display(
    df_sales.limit(20)
)

sale_id,customer_id,vehicle_id,showroom_id,salesperson_id,inventory_id,sale_date,sale_price,purchase_cost,discount,commission,payment_method,created_date,updated_date
1,1,10001,1,1,1,2026-01-10,1180000.00,1050000.00,20000.00,15000.00,Loan,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
2,2,10002,1,2,2,2026-01-20,1350000.00,1200000.00,30000.00,18000.00,Bank Transfer,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
3,3,10008,2,3,3,2026-01-25,1680000.00,1500000.00,25000.00,20000.00,Loan,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
4,5,10016,3,5,5,2026-02-12,830000.00,720000.00,10000.00,9000.00,UPI,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
5,7,10023,4,7,7,2026-02-22,1420000.00,1200000.00,30000.00,16000.00,Loan,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
6,8,10027,5,9,9,2026-03-02,1760000.00,1600000.00,20000.00,18000.00,Bank Transfer,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
7,10,10030,6,11,11,2026-03-12,2450000.00,2200000.00,50000.00,25000.00,Loan,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
8,13,10034,7,13,13,2026-03-25,1690000.00,1550000.00,25000.00,17000.00,UPI,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
9,15,10037,9,15,15,2026-04-05,1880000.00,1700000.00,30000.00,19000.00,Loan,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
10,18,10005,3,6,18,2026-04-18,1080000.00,950000.00,15000.00,10000.00,Cash,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z


In [0]:
# ============================================================
# SALES - CLEAN AND HANDLE NULLS
# ============================================================

# Clean payment method and remove records missing
# required business/relationship fields.
df_sales_clean = (
    df_sales

    # Standardize payment method
    .withColumn(
        "payment_method",
        F.initcap(
            F.trim(F.col("payment_method"))
        )
    )

    # Required identifiers
    .filter(
        F.col("sale_id").isNotNull()
        & F.col("customer_id").isNotNull()
        & F.col("vehicle_id").isNotNull()
        & F.col("showroom_id").isNotNull()
        & F.col("salesperson_id").isNotNull()
        & F.col("inventory_id").isNotNull()
        & F.col("sale_date").isNotNull()
        & F.col("sale_price").isNotNull()
        & F.col("purchase_cost").isNotNull()
    )
)

print(
    "Records after required-field NULL removal:",
    df_sales_clean.count()
)

display(df_sales_clean)

Records after required-field NULL removal: 11


sale_id,customer_id,vehicle_id,showroom_id,salesperson_id,inventory_id,sale_date,sale_price,purchase_cost,discount,commission,payment_method,created_date,updated_date
1,1,10001,1,1,1,2026-01-10,1180000.00,1050000.00,20000.00,15000.00,Loan,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
2,2,10002,1,2,2,2026-01-20,1350000.00,1200000.00,30000.00,18000.00,Bank Transfer,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
3,3,10008,2,3,3,2026-01-25,1680000.00,1500000.00,25000.00,20000.00,Loan,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
4,5,10016,3,5,5,2026-02-12,830000.00,720000.00,10000.00,9000.00,Upi,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
5,7,10023,4,7,7,2026-02-22,1420000.00,1200000.00,30000.00,16000.00,Loan,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
6,8,10027,5,9,9,2026-03-02,1760000.00,1600000.00,20000.00,18000.00,Bank Transfer,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
7,10,10030,6,11,11,2026-03-12,2450000.00,2200000.00,50000.00,25000.00,Loan,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
8,13,10034,7,13,13,2026-03-25,1690000.00,1550000.00,25000.00,17000.00,Upi,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
9,15,10037,9,15,15,2026-04-05,1880000.00,1700000.00,30000.00,19000.00,Loan,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z
10,18,10005,3,6,18,2026-04-18,1080000.00,950000.00,15000.00,10000.00,Cash,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z


In [0]:
# ============================================================
# SALES - FINANCIAL VALIDATION
# ============================================================

# Sale price and purchase cost cannot be negative.
invalid_prices = (
    df_sales_clean
    .filter(
        (F.col("sale_price") < 0) |
        (F.col("purchase_cost") < 0)
    )
)

print(
    "Records with invalid prices:",
    invalid_prices.count()
)

# Discount and commission should also not be negative.
invalid_adjustments = (
    df_sales_clean
    .filter(
        (F.col("discount") < 0) |
        (F.col("commission") < 0)
    )
)

print(
    "Records with negative discount/commission:",
    invalid_adjustments.count()
)

display(invalid_prices)
display(invalid_adjustments)

Records with invalid prices: 0
Records with negative discount/commission: 0


sale_id,customer_id,vehicle_id,showroom_id,salesperson_id,inventory_id,sale_date,sale_price,purchase_cost,discount,commission,payment_method,created_date,updated_date


sale_id,customer_id,vehicle_id,showroom_id,salesperson_id,inventory_id,sale_date,sale_price,purchase_cost,discount,commission,payment_method,created_date,updated_date


In [0]:
# ============================================================
# SALES - REFERENTIAL INTEGRITY CHECK
# ============================================================

# Load existing Silver reference tables
df_customer_silver = spark.table(
    "showroom_analytics.silver.customer"
)

df_showroom_silver = spark.table(
    "showroom_analytics.silver.showroom"
)

df_salesperson_silver = spark.table(
    "showroom_analytics.silver.salesperson"
)

df_inventory_silver = spark.table(
    "showroom_analytics.silver.inventory"
)

# Check customer
invalid_customers = (
    df_sales_clean
    .join(
        df_customer_silver.select("customer_id"),
        "customer_id",
        "left_anti"
    )
)

# Check showroom
invalid_showrooms = (
    df_sales_clean
    .join(
        df_showroom_silver.select("showroom_id"),
        "showroom_id",
        "left_anti"
    )
)

# Check salesperson
invalid_salespersons = (
    df_sales_clean
    .join(
        df_salesperson_silver.select("salesperson_id"),
        "salesperson_id",
        "left_anti"
    )
)

# Check inventory
invalid_inventory = (
    df_sales_clean
    .join(
        df_inventory_silver.select("inventory_id"),
        "inventory_id",
        "left_anti"
    )
)

print("Invalid customer_id:", invalid_customers.count())
print("Invalid showroom_id:", invalid_showrooms.count())
print("Invalid salesperson_id:", invalid_salespersons.count())
print("Invalid inventory_id:", invalid_inventory.count())

Invalid customer_id: 0
Invalid showroom_id: 0
Invalid salesperson_id: 0
Invalid inventory_id: 0


In [0]:
# ============================================================
# SALES - DEDUPLICATION
# ============================================================

# Keep the latest version of each sale.
sales_window = (
    Window
    .partitionBy("sale_id")
    .orderBy(
        F.col("updated_date").desc_nulls_last()
    )
)

df_sales_clean = (
    df_sales_clean
    .withColumn(
        "_row_number",
        F.row_number().over(sales_window)
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)

print(
    "Records after deduplication:",
    df_sales_clean.count()
)

Records after deduplication: 11


In [0]:
# ============================================================
# SALES - SILVER PROCESSING METADATA
# ============================================================

# Store when the Silver layer processed the sale.
df_sales_clean = (
    df_sales_clean
    .withColumn(
        "_silver_processed_timestamp",
        F.current_timestamp()
    )
)

In [0]:
# ============================================================
# SALES - WRITE TO SILVER DELTA
# ============================================================

df_sales_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "showroom_analytics.silver.sales"
    )

print(
    "SUCCESS: sales written to Silver."
)

SUCCESS: sales written to Silver.


In [0]:
# ============================================================
# SALES - VERIFY SILVER
# ============================================================

df_silver_sales = spark.table(
    "showroom_analytics.silver.sales"
)

display(df_silver_sales)

print(
    "Silver sales records:",
    df_silver_sales.count()
)

df_silver_sales.printSchema()

sale_id,customer_id,vehicle_id,showroom_id,salesperson_id,inventory_id,sale_date,sale_price,purchase_cost,discount,commission,payment_method,created_date,updated_date,_silver_processed_timestamp
1,1,10001,1,1,1,2026-01-10,1180000.00,1050000.00,20000.00,15000.00,Loan,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z,2026-08-17T08:33:27.851Z
2,2,10002,1,2,2,2026-01-20,1350000.00,1200000.00,30000.00,18000.00,Bank Transfer,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z,2026-08-17T08:33:27.851Z
3,3,10008,2,3,3,2026-01-25,1680000.00,1500000.00,25000.00,20000.00,Loan,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z,2026-08-17T08:33:27.851Z
4,5,10016,3,5,5,2026-02-12,830000.00,720000.00,10000.00,9000.00,Upi,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z,2026-08-17T08:33:27.851Z
5,7,10023,4,7,7,2026-02-22,1420000.00,1200000.00,30000.00,16000.00,Loan,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z,2026-08-17T08:33:27.851Z
6,8,10027,5,9,9,2026-03-02,1760000.00,1600000.00,20000.00,18000.00,Bank Transfer,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z,2026-08-17T08:33:27.851Z
7,10,10030,6,11,11,2026-03-12,2450000.00,2200000.00,50000.00,25000.00,Loan,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z,2026-08-17T08:33:27.851Z
8,13,10034,7,13,13,2026-03-25,1690000.00,1550000.00,25000.00,17000.00,Upi,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z,2026-08-17T08:33:27.851Z
9,15,10037,9,15,15,2026-04-05,1880000.00,1700000.00,30000.00,19000.00,Loan,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z,2026-08-17T08:33:27.851Z
10,18,10005,3,6,18,2026-04-18,1080000.00,950000.00,15000.00,10000.00,Cash,2026-08-15T16:15:42.781Z,2026-08-15T16:15:42.781Z,2026-08-17T08:33:27.851Z


Silver sales records: 11
root
 |-- sale_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- vehicle_id: integer (nullable = true)
 |-- showroom_id: integer (nullable = true)
 |-- salesperson_id: integer (nullable = true)
 |-- inventory_id: long (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- sale_price: decimal(18,2) (nullable = true)
 |-- purchase_cost: decimal(18,2) (nullable = true)
 |-- discount: decimal(18,2) (nullable = true)
 |-- commission: decimal(18,2) (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- updated_date: timestamp (nullable = true)
 |-- _silver_processed_timestamp: timestamp (nullable = true)



In [0]:
# ============================================================
# EXPENSES - READ BRONZE DATA
# ============================================================

# Path to the expenses Bronze folder
expenses_path = (
    bronze_mssql_path + "expenses/"
)

# Read Parquet data from Bronze
df_expenses = spark.read.parquet(
    expenses_path
)

# Display source data
display(df_expenses)

# Display schema
df_expenses.printSchema()

# Display record count
print(
    "Total expense records:",
    df_expenses.count()
)

expense_id,showroom_id,expense_date,expense_type,amount,description,created_date,updated_date
1,1,2026-01-31,Rent,180000.00,Monthly showroom rent,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z
2,1,2026-01-31,Electricity,32000.00,Monthly electricity bill,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z
3,2,2026-01-31,Marketing,75000.00,Digital marketing expense,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z
4,2,2026-02-28,Maintenance,28000.00,Showroom maintenance,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z
5,3,2026-02-28,Rent,150000.00,Monthly showroom rent,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z
6,3,2026-02-28,Salary,220000.00,Sales staff salary,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z
7,4,2026-03-31,Electricity,35000.00,Monthly electricity bill,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z
8,4,2026-03-31,Marketing,60000.00,Local marketing campaign,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z
9,5,2026-03-31,Rent,175000.00,Monthly showroom rent,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z
10,5,2026-04-30,Maintenance,24000.00,Vehicle display area maintenance,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z


root
 |-- expense_id: long (nullable = true)
 |-- showroom_id: integer (nullable = true)
 |-- expense_date: date (nullable = true)
 |-- expense_type: string (nullable = true)
 |-- amount: decimal(18,2) (nullable = true)
 |-- description: string (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- updated_date: timestamp (nullable = true)

Total expense records: 15


In [0]:
# ============================================================
# EXPENSES - CLEAN AND HANDLE NULLS
# ============================================================

# Clean text fields and remove records missing
# required business fields.
df_expenses_clean = (
    df_expenses

    # Standardize expense type
    .withColumn(
        "expense_type",
        F.initcap(
            F.trim(F.col("expense_type"))
        )
    )

    # Remove unnecessary spaces from description
    .withColumn(
        "description",
        F.trim(F.col("description"))
    )

    # Required fields must not be NULL
    .filter(
        F.col("expense_id").isNotNull()
        & F.col("showroom_id").isNotNull()
        & F.col("expense_date").isNotNull()
        & F.col("amount").isNotNull()
    )
)

print(
    "Records after required-field NULL removal:",
    df_expenses_clean.count()
)

display(df_expenses_clean)

Records after required-field NULL removal: 15


expense_id,showroom_id,expense_date,expense_type,amount,description,created_date,updated_date
1,1,2026-01-31,Rent,180000.00,Monthly showroom rent,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z
2,1,2026-01-31,Electricity,32000.00,Monthly electricity bill,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z
3,2,2026-01-31,Marketing,75000.00,Digital marketing expense,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z
4,2,2026-02-28,Maintenance,28000.00,Showroom maintenance,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z
5,3,2026-02-28,Rent,150000.00,Monthly showroom rent,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z
6,3,2026-02-28,Salary,220000.00,Sales staff salary,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z
7,4,2026-03-31,Electricity,35000.00,Monthly electricity bill,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z
8,4,2026-03-31,Marketing,60000.00,Local marketing campaign,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z
9,5,2026-03-31,Rent,175000.00,Monthly showroom rent,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z
10,5,2026-04-30,Maintenance,24000.00,Vehicle display area maintenance,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z


In [0]:
# ============================================================
# EXPENSES - FINANCIAL VALIDATION
# ============================================================

# Expense amount cannot be negative.
invalid_expenses = (
    df_expenses_clean
    .filter(
        F.col("amount") < 0
    )
)

print(
    "Expenses with negative amount:",
    invalid_expenses.count()
)

display(invalid_expenses)

Expenses with negative amount: 0


expense_id,showroom_id,expense_date,expense_type,amount,description,created_date,updated_date


In [0]:
# ============================================================
# EXPENSES - SHOWROOM REFERENTIAL INTEGRITY
# ============================================================

# Load the existing Silver showroom table
df_showroom_silver = spark.table(
    "showroom_analytics.silver.showroom"
)

# Find expenses belonging to a showroom
# that does not exist in Silver.
invalid_showrooms = (
    df_expenses_clean
    .join(
        df_showroom_silver.select("showroom_id"),
        on="showroom_id",
        how="left_anti"
    )
)

print(
    "Expenses with invalid showroom_id:",
    invalid_showrooms.count()
)

display(invalid_showrooms)

Expenses with invalid showroom_id: 0


showroom_id,expense_id,expense_date,expense_type,amount,description,created_date,updated_date


In [0]:
# ============================================================
# EXPENSES - DEDUPLICATION
# ============================================================

# Keep the latest version of each expense.
expenses_window = (
    Window
    .partitionBy("expense_id")
    .orderBy(
        F.col("updated_date").desc_nulls_last()
    )
)

df_expenses_clean = (
    df_expenses_clean
    .withColumn(
        "_row_number",
        F.row_number().over(expenses_window)
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)

print(
    "Records after deduplication:",
    df_expenses_clean.count()
)

Records after deduplication: 15


In [0]:
# ============================================================
# EXPENSES - SILVER PROCESSING METADATA
# ============================================================

# Store the timestamp when the Silver layer processed
# the expense record.
df_expenses_clean = (
    df_expenses_clean
    .withColumn(
        "_silver_processed_timestamp",
        F.current_timestamp()
    )
)

In [0]:
# ============================================================
# EXPENSES - WRITE TO SILVER DELTA
# ============================================================

df_expenses_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "showroom_analytics.silver.expenses"
    )

print(
    "SUCCESS: expenses written to Silver."
)

SUCCESS: expenses written to Silver.


In [0]:
# ============================================================
# EXPENSES - VERIFY SILVER
# ============================================================

df_silver_expenses = spark.table(
    "showroom_analytics.silver.expenses"
)

display(df_silver_expenses)

print(
    "Silver expenses records:",
    df_silver_expenses.count()
)

df_silver_expenses.printSchema()

expense_id,showroom_id,expense_date,expense_type,amount,description,created_date,updated_date,_silver_processed_timestamp
1,1,2026-01-31,Rent,180000.00,Monthly showroom rent,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z,2026-08-17T08:35:50.036Z
2,1,2026-01-31,Electricity,32000.00,Monthly electricity bill,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z,2026-08-17T08:35:50.036Z
3,2,2026-01-31,Marketing,75000.00,Digital marketing expense,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z,2026-08-17T08:35:50.036Z
4,2,2026-02-28,Maintenance,28000.00,Showroom maintenance,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z,2026-08-17T08:35:50.036Z
5,3,2026-02-28,Rent,150000.00,Monthly showroom rent,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z,2026-08-17T08:35:50.036Z
6,3,2026-02-28,Salary,220000.00,Sales staff salary,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z,2026-08-17T08:35:50.036Z
7,4,2026-03-31,Electricity,35000.00,Monthly electricity bill,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z,2026-08-17T08:35:50.036Z
8,4,2026-03-31,Marketing,60000.00,Local marketing campaign,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z,2026-08-17T08:35:50.036Z
9,5,2026-03-31,Rent,175000.00,Monthly showroom rent,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z,2026-08-17T08:35:50.036Z
10,5,2026-04-30,Maintenance,24000.00,Vehicle display area maintenance,2026-08-15T16:15:42.801Z,2026-08-15T16:15:42.801Z,2026-08-17T08:35:50.036Z


Silver expenses records: 15
root
 |-- expense_id: long (nullable = true)
 |-- showroom_id: integer (nullable = true)
 |-- expense_date: date (nullable = true)
 |-- expense_type: string (nullable = true)
 |-- amount: decimal(18,2) (nullable = true)
 |-- description: string (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- updated_date: timestamp (nullable = true)
 |-- _silver_processed_timestamp: timestamp (nullable = true)



In [0]:
# ============================================================
# CAMPAIGN EXPENSE - READ BRONZE DATA
# ============================================================

# Path to the campaign_expense Bronze folder
campaign_expense_path = (
    bronze_mssql_path + "campaign_expense/"
)

# Read Parquet data from Bronze
df_campaign_expense = spark.read.parquet(
    campaign_expense_path
)

# Display source data
display(df_campaign_expense)

# Display schema
df_campaign_expense.printSchema()

# Display record count
print(
    "Total campaign expense records:",
    df_campaign_expense.count()
)

campaign_expense_id,campaign_id,showroom_id,expense_date,amount,created_date,updated_date
1,1,1,2025-10-15,120000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
2,1,2,2025-10-20,150000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
3,2,1,2025-11-10,85000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
4,2,6,2025-11-15,90000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
5,3,5,2025-12-10,65000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
6,3,2,2025-12-15,70000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
7,4,1,2026-01-05,110000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
8,4,3,2026-01-08,95000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
9,5,4,2026-01-18,70000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
10,5,7,2026-01-20,65000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z


root
 |-- campaign_expense_id: long (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- showroom_id: integer (nullable = true)
 |-- expense_date: date (nullable = true)
 |-- amount: decimal(18,2) (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- updated_date: timestamp (nullable = true)

Total campaign expense records: 15


In [0]:
# ============================================================
# CAMPAIGN EXPENSE - CLEAN AND HANDLE NULLS
# ============================================================

df_campaign_expense_clean = (
    df_campaign_expense

    # Required fields must be present
    .filter(
        F.col("campaign_expense_id").isNotNull()
        & F.col("campaign_id").isNotNull()
        & F.col("showroom_id").isNotNull()
        & F.col("expense_date").isNotNull()
        & F.col("amount").isNotNull()
    )
)

print(
    "Records after required-field NULL removal:",
    df_campaign_expense_clean.count()
)

display(df_campaign_expense_clean)

Records after required-field NULL removal: 15


campaign_expense_id,campaign_id,showroom_id,expense_date,amount,created_date,updated_date
1,1,1,2025-10-15,120000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
2,1,2,2025-10-20,150000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
3,2,1,2025-11-10,85000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
4,2,6,2025-11-15,90000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
5,3,5,2025-12-10,65000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
6,3,2,2025-12-15,70000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
7,4,1,2026-01-05,110000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
8,4,3,2026-01-08,95000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
9,5,4,2026-01-18,70000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
10,5,7,2026-01-20,65000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z


In [0]:
# ============================================================
# CAMPAIGN EXPENSE - REFERENTIAL INTEGRITY
# ============================================================

df_campaign_silver = spark.table(
    "showroom_analytics.silver.campaign"
)

df_showroom_silver = spark.table(
    "showroom_analytics.silver.showroom"
)

# Check campaign references
invalid_campaigns = (
    df_campaign_expense_clean
    .join(
        df_campaign_silver.select("campaign_id"),
        on="campaign_id",
        how="left_anti"
    )
)

# Check showroom references
invalid_showrooms = (
    df_campaign_expense_clean
    .join(
        df_showroom_silver.select("showroom_id"),
        on="showroom_id",
        how="left_anti"
    )
)

print(
    "Invalid campaign_id:",
    invalid_campaigns.count()
)

print(
    "Invalid showroom_id:",
    invalid_showrooms.count()
)

Invalid campaign_id: 0
Invalid showroom_id: 0


In [0]:
# ============================================================
# CAMPAIGN EXPENSE - FINANCIAL VALIDATION
# ============================================================

# Expense amount cannot be negative.
invalid_amounts = (
    df_campaign_expense_clean
    .filter(
        F.col("amount") < 0
    )
)

print(
    "Negative campaign expenses:",
    invalid_amounts.count()
)

display(invalid_amounts)

Negative campaign expenses: 0


campaign_expense_id,campaign_id,showroom_id,expense_date,amount,created_date,updated_date


In [0]:
# ============================================================
# CAMPAIGN EXPENSE - DEDUPLICATION
# ============================================================

campaign_expense_window = (
    Window
    .partitionBy("campaign_expense_id")
    .orderBy(
        F.col("updated_date").desc_nulls_last()
    )
)

df_campaign_expense_clean = (
    df_campaign_expense_clean
    .withColumn(
        "_row_number",
        F.row_number().over(campaign_expense_window)
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)

print(
    "Records after deduplication:",
    df_campaign_expense_clean.count()
)

Records after deduplication: 15


In [0]:
# ============================================================
# CAMPAIGN EXPENSE - WRITE TO SILVER
# ============================================================

df_campaign_expense_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "showroom_analytics.silver.campaign_expense"
    )

print(
    "SUCCESS: campaign_expense written to Silver."
)

SUCCESS: campaign_expense written to Silver.


In [0]:
# ============================================================
# CAMPAIGN EXPENSE - VERIFY SILVER
# ============================================================

df_silver_campaign_expense = spark.table(
    "showroom_analytics.silver.campaign_expense"
)

display(df_silver_campaign_expense)

print(
    "Silver campaign_expense records:",
    df_silver_campaign_expense.count()
)

df_silver_campaign_expense.printSchema()

campaign_expense_id,campaign_id,showroom_id,expense_date,amount,created_date,updated_date
1,1,1,2025-10-15,120000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
2,1,2,2025-10-20,150000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
3,2,1,2025-11-10,85000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
4,2,6,2025-11-15,90000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
5,3,5,2025-12-10,65000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
6,3,2,2025-12-15,70000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
7,4,1,2026-01-05,110000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
8,4,3,2026-01-08,95000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
9,5,4,2026-01-18,70000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z
10,5,7,2026-01-20,65000.00,2026-08-15T16:15:42.825Z,2026-08-15T16:15:42.825Z


Silver campaign_expense records: 15
root
 |-- campaign_expense_id: long (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- showroom_id: integer (nullable = true)
 |-- expense_date: date (nullable = true)
 |-- amount: decimal(18,2) (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- updated_date: timestamp (nullable = true)



In [0]:
# ============================================================
# VEHICLE MODEL - READ POSTGRESQL BRONZE DATA
# ============================================================

vehicle_model_path = (
    bronze_postgresql_path + "vehicle_model/"
)

df_vehicle_model = spark.read.parquet(
    vehicle_model_path
)

display(df_vehicle_model)

df_vehicle_model.printSchema()

print(
    "Total vehicle_model records:",
    df_vehicle_model.count()
)

model_id,brand,model_name,vehicle_type,segment,created_date,updated_date
102,Tata,Punch,SUV,Entry,2026-08-15T21:28:25.247Z,2026-08-15T21:28:25.247Z
103,Tata,Harrier,SUV,Premium,2026-08-15T21:28:25.247Z,2026-08-15T21:28:25.247Z
104,Tata,Safari,SUV,Premium,2026-08-15T21:28:25.247Z,2026-08-15T21:28:25.247Z
106,Hyundai,Venue,SUV,Entry,2026-08-15T21:28:25.247Z,2026-08-15T21:28:25.247Z
107,Hyundai,Verna,Sedan,Mid,2026-08-15T21:28:25.247Z,2026-08-15T21:28:25.247Z
108,Hyundai,i20,Hatchback,Entry,2026-08-15T21:28:25.247Z,2026-08-15T21:28:25.247Z
109,Maruti,Swift,Hatchback,Entry,2026-08-15T21:28:25.247Z,2026-08-15T21:28:25.247Z
110,Maruti,Baleno,Hatchback,Mid,2026-08-15T21:28:25.247Z,2026-08-15T21:28:25.247Z
111,Maruti,Brezza,SUV,Mid,2026-08-15T21:28:25.247Z,2026-08-15T21:28:25.247Z
112,Maruti,Grand Vitara,SUV,Premium,2026-08-15T21:28:25.247Z,2026-08-15T21:28:25.247Z


root
 |-- model_id: integer (nullable = true)
 |-- brand: string (nullable = true)
 |-- model_name: string (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- updated_date: timestamp (nullable = true)

Total vehicle_model records: 28


In [0]:
# ============================================================
# VEHICLE MODEL - CLEAN AND HANDLE NULLS
# ============================================================

df_vehicle_model_clean = (
    df_vehicle_model

    # Clean text columns
    .withColumn(
        "brand",
        F.initcap(F.trim(F.col("brand")))
    )
    .withColumn(
        "model_name",
        F.trim(F.col("model_name"))
    )
    .withColumn(
        "vehicle_type",
        F.initcap(F.trim(F.col("vehicle_type")))
    )
    .withColumn(
        "segment",
        F.initcap(F.trim(F.col("segment")))
    )

    # These fields are required for a valid vehicle model
    .filter(
        F.col("model_id").isNotNull()
        & F.col("brand").isNotNull()
        & F.col("model_name").isNotNull()
        & F.col("vehicle_type").isNotNull()
        & F.col("segment").isNotNull()
    )
)

print(
    "Records after required NULL removal:",
    df_vehicle_model_clean.count()
)

display(df_vehicle_model_clean)

Records after required NULL removal: 28


model_id,brand,model_name,vehicle_type,segment,created_date,updated_date
102,Tata,Punch,Suv,Entry,2026-08-15T21:28:25.247Z,2026-08-15T21:28:25.247Z
103,Tata,Harrier,Suv,Premium,2026-08-15T21:28:25.247Z,2026-08-15T21:28:25.247Z
104,Tata,Safari,Suv,Premium,2026-08-15T21:28:25.247Z,2026-08-15T21:28:25.247Z
106,Hyundai,Venue,Suv,Entry,2026-08-15T21:28:25.247Z,2026-08-15T21:28:25.247Z
107,Hyundai,Verna,Sedan,Mid,2026-08-15T21:28:25.247Z,2026-08-15T21:28:25.247Z
108,Hyundai,i20,Hatchback,Entry,2026-08-15T21:28:25.247Z,2026-08-15T21:28:25.247Z
109,Maruti,Swift,Hatchback,Entry,2026-08-15T21:28:25.247Z,2026-08-15T21:28:25.247Z
110,Maruti,Baleno,Hatchback,Mid,2026-08-15T21:28:25.247Z,2026-08-15T21:28:25.247Z
111,Maruti,Brezza,Suv,Mid,2026-08-15T21:28:25.247Z,2026-08-15T21:28:25.247Z
112,Maruti,Grand Vitara,Suv,Premium,2026-08-15T21:28:25.247Z,2026-08-15T21:28:25.247Z


In [0]:
# ============================================================
# VEHICLE MODEL - DUPLICATE CHECK
# ============================================================

df_vehicle_model_clean \
    .groupBy("model_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

+--------+-----+
|model_id|count|
+--------+-----+
+--------+-----+



In [0]:
# ============================================================
# VEHICLE MODEL - DEDUPLICATION
# ============================================================

vehicle_model_window = (
    Window
    .partitionBy("model_id")
    .orderBy(
        F.col("updated_date").desc_nulls_last()
    )
)

df_vehicle_model_clean = (
    df_vehicle_model_clean
    .withColumn(
        "_row_number",
        F.row_number().over(vehicle_model_window)
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)

print(
    "Records after deduplication:",
    df_vehicle_model_clean.count()
)

Records after deduplication: 28


In [0]:
# ============================================================
# VEHICLE MODEL - SILVER METADATA
# ============================================================

df_vehicle_model_clean = (
    df_vehicle_model_clean
    .withColumn(
        "_silver_processed_timestamp",
        F.current_timestamp()
    )
)

In [0]:
# ============================================================
# VEHICLE MODEL - WRITE TO SILVER
# ============================================================

df_vehicle_model_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "showroom_analytics.silver.vehicle_model"
    )

print(
    "SUCCESS: vehicle_model written to Silver."
)

SUCCESS: vehicle_model written to Silver.


In [0]:
# ============================================================
# VEHICLE - READ POSTGRESQL BRONZE DATA
# ============================================================

vehicle_path = (
    bronze_postgresql_path + "vehicle/"
)

df_vehicle = spark.read.parquet(
    vehicle_path
)

display(df_vehicle)

df_vehicle.printSchema()

print(
    "Total vehicle records:",
    df_vehicle.count()
)

vehicle_id,model_id,variant,fuel_type,transmission,manufacturing_year,base_price,created_date,updated_date
10001,101,XZ+,Petrol,Manual,2026,1250000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z
10002,101,XZ+ Lux,Petrol,Automatic,2026,1400000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z
10003,101,XZ+ Diesel,Diesel,Manual,2026,1450000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z
10004,102,Adventure,Petrol,Manual,2026,950000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z
10005,102,Creative+,Petrol,Automatic,2026,1100000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z
10006,103,XZA,Diesel,Automatic,2026,2100000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z
10007,104,Accomplished,Diesel,Automatic,2026,2400000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z
10008,105,S,Petrol,Manual,2026,1450000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z
10009,105,SX,Petrol,Automatic,2026,1750000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z
10010,105,SX Diesel,Diesel,Automatic,2026,1900000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z


root
 |-- vehicle_id: integer (nullable = true)
 |-- model_id: integer (nullable = true)
 |-- variant: string (nullable = true)
 |-- fuel_type: string (nullable = true)
 |-- transmission: string (nullable = true)
 |-- manufacturing_year: integer (nullable = true)
 |-- base_price: decimal(38,18) (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- updated_date: timestamp (nullable = true)

Total vehicle records: 38


In [0]:
# ============================================================
# VEHICLE - CLEAN AND HANDLE NULLS
# ============================================================

df_vehicle_clean = (
    df_vehicle

    # Clean text columns
    .withColumn(
        "variant",
        F.trim(F.col("variant"))
    )
    .withColumn(
        "fuel_type",
        F.initcap(F.trim(F.col("fuel_type")))
    )
    .withColumn(
        "transmission",
        F.initcap(F.trim(F.col("transmission")))
    )

    # Required fields
    .filter(
        F.col("vehicle_id").isNotNull()
        & F.col("model_id").isNotNull()
        & F.col("variant").isNotNull()
        & F.col("fuel_type").isNotNull()
        & F.col("transmission").isNotNull()
        & F.col("manufacturing_year").isNotNull()
        & F.col("base_price").isNotNull()
    )
)

print(
    "Records after required NULL removal:",
    df_vehicle_clean.count()
)

display(df_vehicle_clean)

Records after required NULL removal: 38


vehicle_id,model_id,variant,fuel_type,transmission,manufacturing_year,base_price,created_date,updated_date
10001,101,XZ+,Petrol,Manual,2026,1250000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z
10002,101,XZ+ Lux,Petrol,Automatic,2026,1400000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z
10003,101,XZ+ Diesel,Diesel,Manual,2026,1450000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z
10004,102,Adventure,Petrol,Manual,2026,950000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z
10005,102,Creative+,Petrol,Automatic,2026,1100000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z
10006,103,XZA,Diesel,Automatic,2026,2100000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z
10007,104,Accomplished,Diesel,Automatic,2026,2400000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z
10008,105,S,Petrol,Manual,2026,1450000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z
10009,105,SX,Petrol,Automatic,2026,1750000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z
10010,105,SX Diesel,Diesel,Automatic,2026,1900000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z


In [0]:
# ============================================================
# VEHICLE - MODEL REFERENTIAL INTEGRITY
# ============================================================

df_vehicle_model_silver = spark.table(
    "showroom_analytics.silver.vehicle_model"
)

invalid_models = (
    df_vehicle_clean
    .join(
        df_vehicle_model_silver.select("model_id"),
        on="model_id",
        how="left_anti"
    )
)

print(
    "Vehicles with invalid model_id:",
    invalid_models.count()
)

display(invalid_models)

Vehicles with invalid model_id: 0


model_id,vehicle_id,variant,fuel_type,transmission,manufacturing_year,base_price,created_date,updated_date


In [0]:
# ============================================================
# VEHICLE - BUSINESS VALIDATION
# ============================================================

# Check duplicate vehicle IDs
print("Duplicate vehicle IDs:")

df_vehicle_clean \
    .groupBy("vehicle_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

# Check negative base prices
print("Vehicles with negative base price:")

df_vehicle_clean \
    .filter(
        F.col("base_price") < 0
    ) \
    .show()

# Check invalid manufacturing years
print("Invalid manufacturing years:")

df_vehicle_clean \
    .filter(
        (F.col("manufacturing_year") < 1900) |
        (F.col("manufacturing_year") > 2100)
    ) \
    .show()

Duplicate vehicle IDs:
+----------+-----+
|vehicle_id|count|
+----------+-----+
+----------+-----+

Vehicles with negative base price:
+----------+--------+-------+---------+------------+------------------+----------+------------+------------+
|vehicle_id|model_id|variant|fuel_type|transmission|manufacturing_year|base_price|created_date|updated_date|
+----------+--------+-------+---------+------------+------------------+----------+------------+------------+
+----------+--------+-------+---------+------------+------------------+----------+------------+------------+

Invalid manufacturing years:
+----------+--------+-------+---------+------------+------------------+----------+------------+------------+
|vehicle_id|model_id|variant|fuel_type|transmission|manufacturing_year|base_price|created_date|updated_date|
+----------+--------+-------+---------+------------+------------------+----------+------------+------------+
+----------+--------+-------+---------+------------+------------------+-

In [0]:
# ============================================================
# VEHICLE - DEDUPLICATION
# ============================================================

vehicle_window = (
    Window
    .partitionBy("vehicle_id")
    .orderBy(
        F.col("updated_date").desc_nulls_last()
    )
)

df_vehicle_clean = (
    df_vehicle_clean
    .withColumn(
        "_row_number",
        F.row_number().over(vehicle_window)
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)

print(
    "Records after deduplication:",
    df_vehicle_clean.count()
)

Records after deduplication: 38


In [0]:
# ============================================================
# VEHICLE - SILVER METADATA
# ============================================================

df_vehicle_clean = (
    df_vehicle_clean
    .withColumn(
        "_silver_processed_timestamp",
        F.current_timestamp()
    )
)

In [0]:
# ============================================================
# VEHICLE - WRITE TO SILVER DELTA
# ============================================================

df_vehicle_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "showroom_analytics.silver.vehicle"
    )

print(
    "SUCCESS: vehicle written to Silver."
)

SUCCESS: vehicle written to Silver.


In [0]:
# ============================================================
# VEHICLE - VERIFY SILVER TABLE
# ============================================================

df_silver_vehicle = spark.table(
    "showroom_analytics.silver.vehicle"
)

display(df_silver_vehicle)

print(
    "Silver vehicle records:",
    df_silver_vehicle.count()
)

df_silver_vehicle.printSchema()

vehicle_id,model_id,variant,fuel_type,transmission,manufacturing_year,base_price,created_date,updated_date,_silver_processed_timestamp
10001,101,XZ+,Petrol,Manual,2026,1250000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z,2026-08-17T08:43:28.184Z
10002,101,XZ+ Lux,Petrol,Automatic,2026,1400000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z,2026-08-17T08:43:28.184Z
10003,101,XZ+ Diesel,Diesel,Manual,2026,1450000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z,2026-08-17T08:43:28.184Z
10004,102,Adventure,Petrol,Manual,2026,950000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z,2026-08-17T08:43:28.184Z
10005,102,Creative+,Petrol,Automatic,2026,1100000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z,2026-08-17T08:43:28.184Z
10006,103,XZA,Diesel,Automatic,2026,2100000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z,2026-08-17T08:43:28.184Z
10007,104,Accomplished,Diesel,Automatic,2026,2400000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z,2026-08-17T08:43:28.184Z
10008,105,S,Petrol,Manual,2026,1450000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z,2026-08-17T08:43:28.184Z
10009,105,SX,Petrol,Automatic,2026,1750000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z,2026-08-17T08:43:28.184Z
10010,105,SX Diesel,Diesel,Automatic,2026,1900000.000000000000000000,2026-08-15T21:28:55.566Z,2026-08-15T21:28:55.566Z,2026-08-17T08:43:28.184Z


Silver vehicle records: 38
root
 |-- vehicle_id: integer (nullable = true)
 |-- model_id: integer (nullable = true)
 |-- variant: string (nullable = true)
 |-- fuel_type: string (nullable = true)
 |-- transmission: string (nullable = true)
 |-- manufacturing_year: integer (nullable = true)
 |-- base_price: decimal(38,18) (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- updated_date: timestamp (nullable = true)
 |-- _silver_processed_timestamp: timestamp (nullable = true)



In [0]:
# ============================================================
# INVENTORY - VEHICLE REFERENTIAL INTEGRITY
# ============================================================

df_inventory_silver = spark.table(
    "showroom_analytics.silver.inventory"
)

invalid_inventory_vehicles = (
    df_inventory_silver
    .join(
        df_silver_vehicle.select("vehicle_id"),
        on="vehicle_id",
        how="left_anti"
    )
)

print(
    "Inventory records with invalid vehicle_id:",
    invalid_inventory_vehicles.count()
)

display(invalid_inventory_vehicles)

Inventory records with invalid vehicle_id: 0


vehicle_id,inventory_id,showroom_id,vin,purchase_date,purchase_cost,status,created_date,updated_date,_silver_processed_timestamp


In [0]:
# ============================================================
# SALES - VEHICLE REFERENTIAL INTEGRITY
# ============================================================

df_sales_silver = spark.table(
    "showroom_analytics.silver.sales"
)

invalid_sales_vehicles = (
    df_sales_silver
    .join(
        df_silver_vehicle.select("vehicle_id"),
        on="vehicle_id",
        how="left_anti"
    )
)

print(
    "Sales records with invalid vehicle_id:",
    invalid_sales_vehicles.count()
)

display(invalid_sales_vehicles)

Sales records with invalid vehicle_id: 0


vehicle_id,sale_id,customer_id,showroom_id,salesperson_id,inventory_id,sale_date,sale_price,purchase_cost,discount,commission,payment_method,created_date,updated_date,_silver_processed_timestamp


In [0]:
# ============================================================
# TEST DRIVE - VEHICLE REFERENTIAL INTEGRITY
# ============================================================

df_test_drive_silver = spark.table(
    "showroom_analytics.silver.test_drive"
)

invalid_test_drive_vehicles = (
    df_test_drive_silver
    .join(
        df_silver_vehicle.select("vehicle_id"),
        on="vehicle_id",
        how="left_anti"
    )
)

print(
    "Test drive records with invalid vehicle_id:",
    invalid_test_drive_vehicles.count()
)

display(invalid_test_drive_vehicles)

Test drive records with invalid vehicle_id: 0


vehicle_id,test_drive_id,customer_id,showroom_id,salesperson_id,test_drive_date,test_drive_status,created_date,updated_date,_silver_processed_timestamp
